# Final Project Notebook: Model Implementation, Validation, and Reporting

This notebook implements supervised learning models to classify student review sentiment and demonstrates how different operationalization and preprocessing choices affect results.

## Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import pickle
import re
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)
from nltk.stem import PorterStemmer

np.random.seed(42)
sns.set_style("whitegrid")
import gc  # For memory management

## Load Data

In [ ]:
# Load processed data
df_final = pd.read_csv('../Assets/processed_reviews.csv.gz', compression='gzip')
print(f"Loaded processed_reviews.csv.gz: {len(df_final):,} reviews")

# Load split indices
with open('../Assets/split_indices.json', 'r') as f:
    split_data = json.load(f)
print("Loaded split_indices.json")

# Load domain stopwords (if exists)
try:
    with open('../Assets/domain_stopwords.json', 'r') as f:
        domain_stopwords = set(json.load(f))
    print(f"Loaded domain_stopwords.json: {len(domain_stopwords)} words")
except FileNotFoundError:
    print("No domain_stopwords.json found - using standard stopwords only")
    domain_stopwords = set()

# Recreate train/test splits from saved indices
train_indices = split_data.get('train_indices', [])
test_indices = split_data.get('test_indices', [])

# Validate index bounds against currently loaded dataset.
# If indices are incompatible with this file, regenerate deterministic 70/30 split.
n_rows = len(df_final)
indices_ok = (
    len(train_indices) > 0 and len(test_indices) > 0 and
    min(train_indices + test_indices) >= 0 and
    max(train_indices + test_indices) < n_rows
)

if not indices_ok:
    print("Detected incompatible split indices; regenerating 70/30 train-test split.")
    from sklearn.model_selection import train_test_split
    all_indices = df_final.index.to_numpy()
    y_all = df_final['sentiment']
    train_indices, test_indices = train_test_split(
        all_indices,
        test_size=0.30,
        random_state=42,
        stratify=y_all
    )
    split_data = {
        'train_indices': train_indices.tolist(),
        'test_indices': test_indices.tolist()
    }
    with open('../Assets/split_indices.json', 'w') as f:
        json.dump(split_data, f)
    print("Saved regenerated split_indices.json (train/test only).")

X_train = df_final.iloc[train_indices]['text_final']
y_train = df_final.iloc[train_indices]['sentiment']

X_test = df_final.iloc[test_indices]['text_final']
y_test = df_final.iloc[test_indices]['sentiment']

n_train = len(X_train)
n_test = len(X_test)

print(f"\nTraining:   {len(X_train):,} reviews")
print(f"Test:       {len(X_test):,} reviews")
print(f"\nPositive: {(y_train==1).sum():,} | Negative: {(y_train==0).sum():,}")
print(f"Class ratio: {(y_train==1).sum() / (y_train==0).sum():.1f}:1")
print("Evaluation split is test-only.")

## Operationalization of Key Concepts


### TF-IDF
**TF-IDF (Term Frequency-Inverse Document Frequency)** balances how often a word appears in a document against how common it is across all documents:
- **TF**: Words appearing more in a review get higher weight
- **IDF**: Words appearing in many reviews get downweighted (e.g., "professor" appears everywhere)
- This helps identify distinctive words that differentiate positive from negative reviews


### Vocabulary Decisions
- **max_features**: Limit to most frequent words (prevents overfitting on rare words)
- **min_df**: Ignore words appearing in fewer than N documents (removes typos, rare terms)
- **max_df**: Ignore words appearing in more than X% of documents (removes universal words)
- **ngram_range**: (1,2) includes both single words and two-word phrases (e.g., "not good")

### Domain Stopwords
Beyond standard stopwords ("the," "is"), we identified 104 domain-specific common words ("class," "professor") that appear frequently in both positive and negative reviews and thus don't help distinguish sentiment.

## Dependent Variable Definition and Coding Rationale

- **Dependent variable**: binary review sentiment `Y`.
- **Coding rule**: `Y=1` for ratings `{4,5}`, `Y=0` for ratings `{1,2}`, and rating `3` is excluded as neutral/ambiguous.
- **Why binary**: supports interpretable, actionable polarity (positive vs negative teaching signals) and reduces midpoint ambiguity.
- **Tradeoff**: collapsing ratings discards ordinal detail, so a non-binary sensitivity check is included in Week 6.

## Controlled Experiments: How Preprocessing Choices Affect Results

We conduct controlled experiments to understand how different operationalization decisions impact model performance.

### Experiment 1: Effect of Domain Stopwords

**Question**: Do domain-specific stopwords improve performance by removing uninformative words?

In [ ]:
# Store results for comparison
exp1_results = []

# Configuration 1: Without domain stopwords
print("Configuration 1: Standard stopwords only")

tfidf_no_domain = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.8,
    stop_words=list(ENGLISH_STOP_WORDS),
    token_pattern=r'\b[a-zA-Z]{3,}\b'
)

X_train_no_domain = tfidf_no_domain.fit_transform(X_train)
X_test_no_domain = tfidf_no_domain.transform(X_test)

# Train Logistic Regression
lr_no_domain = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced', C=1.0)
lr_no_domain.fit(X_train_no_domain, y_train)
y_pred_lr_no_domain = lr_no_domain.predict(X_test_no_domain)

acc_lr_no_domain = accuracy_score(y_test, y_pred_lr_no_domain)
f1_lr_no_domain = f1_score(y_test, y_pred_lr_no_domain)
print(f"Logistic Regression - Accuracy: {acc_lr_no_domain:.4f}, F1-Score: {f1_lr_no_domain:.4f}")

# Train Naive Bayes
nb_no_domain = MultinomialNB(alpha=1.0)
nb_no_domain.fit(X_train_no_domain, y_train)
y_pred_nb_no_domain = nb_no_domain.predict(X_test_no_domain)

acc_nb_no_domain = accuracy_score(y_test, y_pred_nb_no_domain)
f1_nb_no_domain = f1_score(y_test, y_pred_nb_no_domain)
print(f"Naive Bayes        - Accuracy: {acc_nb_no_domain:.4f}, F1-Score: {f1_nb_no_domain:.4f}")

exp1_results.append({
    'config': 'Standard stopwords only',
    'model': 'Logistic Regression',
    'accuracy': acc_lr_no_domain,
    'f1': f1_lr_no_domain
})
exp1_results.append({
    'config': 'Standard stopwords only',
    'model': 'Naive Bayes',
    'accuracy': acc_nb_no_domain,
    'f1': f1_nb_no_domain
})

# Configuration 2: With domain stopwords
print("Configuration 2: Standard + domain stopwords")

combined_stopwords = list(ENGLISH_STOP_WORDS.union(domain_stopwords))

tfidf_with_domain = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.8,
    stop_words=combined_stopwords,
    token_pattern=r'\b[a-zA-Z]{3,}\b'
)

X_train_with_domain = tfidf_with_domain.fit_transform(X_train)
X_test_with_domain = tfidf_with_domain.transform(X_test)

# Train Logistic Regression
lr_with_domain = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced', C=1.0)
lr_with_domain.fit(X_train_with_domain, y_train)
y_pred_lr_with_domain = lr_with_domain.predict(X_test_with_domain)

acc_lr_with_domain = accuracy_score(y_test, y_pred_lr_with_domain)
f1_lr_with_domain = f1_score(y_test, y_pred_lr_with_domain)
print(f"Logistic Regression - Accuracy: {acc_lr_with_domain:.4f}, F1-Score: {f1_lr_with_domain:.4f}")

# Train Naive Bayes
nb_with_domain = MultinomialNB(alpha=1.0)
nb_with_domain.fit(X_train_with_domain, y_train)
y_pred_nb_with_domain = nb_with_domain.predict(X_test_with_domain)

acc_nb_with_domain = accuracy_score(y_test, y_pred_nb_with_domain)
f1_nb_with_domain = f1_score(y_test, y_pred_nb_with_domain)
print(f"Naive Bayes        - Accuracy: {acc_nb_with_domain:.4f}, F1-Score: {f1_nb_with_domain:.4f}")

exp1_results.append({
    'config': 'Standard + domain stopwords',
    'model': 'Logistic Regression',
    'accuracy': acc_lr_with_domain,
    'f1': f1_lr_with_domain
})
exp1_results.append({
    'config': 'Standard + domain stopwords',
    'model': 'Naive Bayes',
    'accuracy': acc_nb_with_domain,
    'f1': f1_nb_with_domain
})

print("EXPERIMENT 1 SUMMARY")
print(f"Logistic Regression:")
print(f"  Accuracy difference: {acc_lr_with_domain - acc_lr_no_domain:+.4f}")
print(f"  F1-Score difference: {f1_lr_with_domain - f1_lr_no_domain:+.4f}")
print(f"\nNaive Bayes:")
print(f"  Accuracy difference: {acc_nb_with_domain - acc_nb_no_domain:+.4f}")
print(f"  F1-Score difference: {f1_nb_with_domain - f1_nb_no_domain:+.4f}")
print("\nInterpretation: Domain stopwords help the model focus on sentiment-bearing")
print("words rather than generic course-related terms. Both models show similar patterns.")

### Experiment 2: Vocabulary Size Comparison

**Question**: Does a larger vocabulary (10,000 vs 5,000 features) improve performance?

In [ ]:
exp2_results = []

# Configuration 1: 5,000 features
print("Configuration 1: 5,000 max features")

tfidf_5k = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.8,
    stop_words=combined_stopwords,
    token_pattern=r'\b[a-zA-Z]{3,}\b'
)

X_train_5k = tfidf_5k.fit_transform(X_train)
X_test_5k = tfidf_5k.transform(X_test)

# Train Logistic Regression
lr_5k = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced', C=1.0)
lr_5k.fit(X_train_5k, y_train)
y_pred_lr_5k = lr_5k.predict(X_test_5k)

acc_lr_5k = accuracy_score(y_test, y_pred_lr_5k)
f1_lr_5k = f1_score(y_test, y_pred_lr_5k)
print(f"Vocabulary size: {len(tfidf_5k.vocabulary_):,}")
print(f"Logistic Regression - Accuracy: {acc_lr_5k:.4f}, F1-Score: {f1_lr_5k:.4f}")

# Train Naive Bayes
nb_5k = MultinomialNB(alpha=1.0)
nb_5k.fit(X_train_5k, y_train)
y_pred_nb_5k = nb_5k.predict(X_test_5k)

acc_nb_5k = accuracy_score(y_test, y_pred_nb_5k)
f1_nb_5k = f1_score(y_test, y_pred_nb_5k)
print(f"Naive Bayes        - Accuracy: {acc_nb_5k:.4f}, F1-Score: {f1_nb_5k:.4f}")

exp2_results.append({
    'config': '5,000 features',
    'model': 'Logistic Regression',
    'vocab_size': len(tfidf_5k.vocabulary_),
    'accuracy': acc_lr_5k,
    'f1': f1_lr_5k
})
exp2_results.append({
    'config': '5,000 features',
    'model': 'Naive Bayes',
    'vocab_size': len(tfidf_5k.vocabulary_),
    'accuracy': acc_nb_5k,
    'f1': f1_nb_5k
})

# Configuration 2: 10,000 features
print("Configuration 2: 10,000 max features")

tfidf_10k = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.8,
    stop_words=combined_stopwords,
    token_pattern=r'\b[a-zA-Z]{3,}\b'
)

X_train_20k = tfidf_10k.fit_transform(X_train)
X_test_20k = tfidf_10k.transform(X_test)

# Train Logistic Regression
lr_20k = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced', C=1.0)
lr_20k.fit(X_train_20k, y_train)
y_pred_lr_20k = lr_20k.predict(X_test_20k)

acc_lr_20k = accuracy_score(y_test, y_pred_lr_20k)
f1_lr_20k = f1_score(y_test, y_pred_lr_20k)
print(f"Vocabulary size: {len(tfidf_10k.vocabulary_):,}")
print(f"Logistic Regression - Accuracy: {acc_lr_20k:.4f}, F1-Score: {f1_lr_20k:.4f}")

# Train Naive Bayes
nb_20k = MultinomialNB(alpha=1.0)
nb_20k.fit(X_train_20k, y_train)
y_pred_nb_20k = nb_20k.predict(X_test_20k)

acc_nb_20k = accuracy_score(y_test, y_pred_nb_20k)
f1_nb_20k = f1_score(y_test, y_pred_nb_20k)
print(f"Naive Bayes        - Accuracy: {acc_nb_20k:.4f}, F1-Score: {f1_nb_20k:.4f}")

exp2_results.append({
    'config': '10,000 features',
    'model': 'Logistic Regression',
    'vocab_size': len(tfidf_10k.vocabulary_),
    'accuracy': acc_lr_20k,
    'f1': f1_lr_20k
})
exp2_results.append({
    'config': '10,000 features',
    'model': 'Naive Bayes',
    'vocab_size': len(tfidf_10k.vocabulary_),
    'accuracy': acc_nb_20k,
    'f1': f1_nb_20k
})

print("EXPERIMENT 2 SUMMARY")
print(f"Logistic Regression:")
print(f"  Accuracy difference: {acc_lr_20k - acc_lr_5k:+.4f}")
print(f"  F1-Score difference: {f1_lr_20k - f1_lr_5k:+.4f}")
print(f"\nNaive Bayes:")
print(f"  Accuracy difference: {acc_nb_20k - acc_nb_5k:+.4f}")
print(f"  F1-Score difference: {f1_nb_20k - f1_nb_5k:+.4f}")
print("\nInterpretation: Larger vocabulary captures more nuanced expressions")
print("but may include noise. The difference shows the trade-off for both models.")

### Experiment 3: N-gram Range (Unigrams vs Unigrams+Bigrams)

**Question**: Does including 2-word phrases improve sentiment detection?

In [ ]:
exp3_results = []

# Configuration 1: Unigrams only
print("Configuration 1: Unigrams only (1,1)")

tfidf_unigram = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 1),
    min_df=5,
    max_df=0.8,
    stop_words=combined_stopwords,
    token_pattern=r'\b[a-zA-Z]{3,}\b'
)

X_train_uni = tfidf_unigram.fit_transform(X_train)
X_test_uni = tfidf_unigram.transform(X_test)

# Train Logistic Regression
lr_uni = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced', C=1.0)
lr_uni.fit(X_train_uni, y_train)
y_pred_lr_uni = lr_uni.predict(X_test_uni)

acc_lr_uni = accuracy_score(y_test, y_pred_lr_uni)
f1_lr_uni = f1_score(y_test, y_pred_lr_uni)
print(f"Logistic Regression - Accuracy: {acc_lr_uni:.4f}, F1-Score: {f1_lr_uni:.4f}")

# Train Naive Bayes
nb_uni = MultinomialNB(alpha=1.0)
nb_uni.fit(X_train_uni, y_train)
y_pred_nb_uni = nb_uni.predict(X_test_uni)

acc_nb_uni = accuracy_score(y_test, y_pred_nb_uni)
f1_nb_uni = f1_score(y_test, y_pred_nb_uni)
print(f"Naive Bayes        - Accuracy: {acc_nb_uni:.4f}, F1-Score: {f1_nb_uni:.4f}")

exp3_results.append({
    'config': 'Unigrams only',
    'model': 'Logistic Regression',
    'accuracy': acc_lr_uni,
    'f1': f1_lr_uni
})
exp3_results.append({
    'config': 'Unigrams only',
    'model': 'Naive Bayes',
    'accuracy': acc_nb_uni,
    'f1': f1_nb_uni
})

# Configuration 2: Unigrams + Bigrams
print("Configuration 2: Unigrams + Bigrams (1,2)")

tfidf_bigram = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.8,
    stop_words=combined_stopwords,
    token_pattern=r'\b[a-zA-Z]{3,}\b'
)

X_train_bi = tfidf_bigram.fit_transform(X_train)
X_test_bi = tfidf_bigram.transform(X_test)

# Train Logistic Regression
lr_bi = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced', C=1.0)
lr_bi.fit(X_train_bi, y_train)
y_pred_lr_bi = lr_bi.predict(X_test_bi)

acc_lr_bi = accuracy_score(y_test, y_pred_lr_bi)
f1_lr_bi = f1_score(y_test, y_pred_lr_bi)
print(f"Logistic Regression - Accuracy: {acc_lr_bi:.4f}, F1-Score: {f1_lr_bi:.4f}")

# Train Naive Bayes
nb_bi = MultinomialNB(alpha=1.0)
nb_bi.fit(X_train_bi, y_train)
y_pred_nb_bi = nb_bi.predict(X_test_bi)

acc_nb_bi = accuracy_score(y_test, y_pred_nb_bi)
f1_nb_bi = f1_score(y_test, y_pred_nb_bi)
print(f"Naive Bayes        - Accuracy: {acc_nb_bi:.4f}, F1-Score: {f1_nb_bi:.4f}")

exp3_results.append({
    'config': 'Unigrams + Bigrams',
    'model': 'Logistic Regression',
    'accuracy': acc_lr_bi,
    'f1': f1_lr_bi
})
exp3_results.append({
    'config': 'Unigrams + Bigrams',
    'model': 'Naive Bayes',
    'accuracy': acc_nb_bi,
    'f1': f1_nb_bi
})

print("EXPERIMENT 3 SUMMARY")
print(f"Logistic Regression:")
print(f"  Accuracy difference: {acc_lr_bi - acc_lr_uni:+.4f}")
print(f"  F1-Score difference: {f1_lr_bi - f1_lr_uni:+.4f}")
print(f"\nNaive Bayes:")
print(f"  Accuracy difference: {acc_nb_bi - acc_nb_uni:+.4f}")
print(f"  F1-Score difference: {f1_nb_bi - f1_nb_uni:+.4f}")
print("\nInterpretation: Bigrams capture phrases like 'not good' or 'very helpful'")
print("that carry sentiment information beyond individual words for both models.")

### Experiment 4: Effect of Stemming

**Question**: Does stemming (reducing words to their root form) improve performance by consolidating word variations?

In [ ]:
exp4_results = []

# Create stemming function
stemmer = PorterStemmer()
token_re = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?")

def stem_text(text):
    if pd.isna(text):
        return ""
    tokens = token_re.findall(str(text).lower())
    return " ".join(stemmer.stem(t) for t in tokens)

# Configuration 1: Without stemming (baseline)
print("Configuration 1: Without stemming")

tfidf_no_stem = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.8,
    stop_words=combined_stopwords,
    token_pattern=r'\b[a-zA-Z]{3,}\b'
)

X_train_no_stem = tfidf_no_stem.fit_transform(X_train)
X_test_no_stem = tfidf_no_stem.transform(X_test)

# Train Logistic Regression
lr_no_stem = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced', C=1.0)
lr_no_stem.fit(X_train_no_stem, y_train)
y_pred_lr_no_stem = lr_no_stem.predict(X_test_no_stem)

acc_lr_no_stem = accuracy_score(y_test, y_pred_lr_no_stem)
f1_lr_no_stem = f1_score(y_test, y_pred_lr_no_stem)
print(f"Vocabulary size: {len(tfidf_no_stem.vocabulary_):,}")
print(f"Logistic Regression - Accuracy: {acc_lr_no_stem:.4f}, F1-Score: {f1_lr_no_stem:.4f}")

# Train Naive Bayes
nb_no_stem = MultinomialNB(alpha=1.0)
nb_no_stem.fit(X_train_no_stem, y_train)
y_pred_nb_no_stem = nb_no_stem.predict(X_test_no_stem)

acc_nb_no_stem = accuracy_score(y_test, y_pred_nb_no_stem)
f1_nb_no_stem = f1_score(y_test, y_pred_nb_no_stem)
print(f"Naive Bayes        - Accuracy: {acc_nb_no_stem:.4f}, F1-Score: {f1_nb_no_stem:.4f}")

exp4_results.append({
    'config': 'Without stemming',
    'model': 'Logistic Regression',
    'vocab_size': len(tfidf_no_stem.vocabulary_),
    'accuracy': acc_lr_no_stem,
    'f1': f1_lr_no_stem
})
exp4_results.append({
    'config': 'Without stemming',
    'model': 'Naive Bayes',
    'vocab_size': len(tfidf_no_stem.vocabulary_),
    'accuracy': acc_nb_no_stem,
    'f1': f1_nb_no_stem
})

# Configuration 2: With stemming
print("Configuration 2: With stemming")

# Apply stemming to all text
print("Applying stemming to training data...")
X_train_stemmed = X_train.apply(stem_text)
X_test_stemmed = X_test.apply(stem_text)

# Show example
print("\nExample of stemming:")
example_idx = X_train.index[0]
print(f"Original: {X_train.iloc[0][:100]}...")
print(f"Stemmed:  {X_train_stemmed.iloc[0][:100]}...")

tfidf_with_stem = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.8,
    stop_words=combined_stopwords,
    token_pattern=r'\b[a-zA-Z]{3,}\b'
)

X_train_stem = tfidf_with_stem.fit_transform(X_train_stemmed)
X_test_stem = tfidf_with_stem.transform(X_test_stemmed)

# Train Logistic Regression
lr_with_stem = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced', C=1.0)
lr_with_stem.fit(X_train_stem, y_train)
y_pred_lr_with_stem = lr_with_stem.predict(X_test_stem)

acc_lr_with_stem = accuracy_score(y_test, y_pred_lr_with_stem)
f1_lr_with_stem = f1_score(y_test, y_pred_lr_with_stem)
print(f"\nVocabulary size: {len(tfidf_with_stem.vocabulary_):,}")
print(f"Logistic Regression - Accuracy: {acc_lr_with_stem:.4f}, F1-Score: {f1_lr_with_stem:.4f}")

# Train Naive Bayes
nb_with_stem = MultinomialNB(alpha=1.0)
nb_with_stem.fit(X_train_stem, y_train)
y_pred_nb_with_stem = nb_with_stem.predict(X_test_stem)

acc_nb_with_stem = accuracy_score(y_test, y_pred_nb_with_stem)
f1_nb_with_stem = f1_score(y_test, y_pred_nb_with_stem)
print(f"Naive Bayes        - Accuracy: {acc_nb_with_stem:.4f}, F1-Score: {f1_nb_with_stem:.4f}")

exp4_results.append({
    'config': 'With stemming',
    'model': 'Logistic Regression',
    'vocab_size': len(tfidf_with_stem.vocabulary_),
    'accuracy': acc_lr_with_stem,
    'f1': f1_lr_with_stem
})
exp4_results.append({
    'config': 'With stemming',
    'model': 'Naive Bayes',
    'vocab_size': len(tfidf_with_stem.vocabulary_),
    'accuracy': acc_nb_with_stem,
    'f1': f1_nb_with_stem
})

print("EXPERIMENT 4 SUMMARY")
print(f"Vocabulary reduction: {len(tfidf_no_stem.vocabulary_) - len(tfidf_with_stem.vocabulary_):,} words")
print(f"\nLogistic Regression:")
print(f"  Accuracy difference: {acc_lr_with_stem - acc_lr_no_stem:+.4f}")
print(f"  F1-Score difference: {f1_lr_with_stem - f1_lr_no_stem:+.4f}")
print(f"\nNaive Bayes:")
print(f"  Accuracy difference: {acc_nb_with_stem - acc_nb_no_stem:+.4f}")
print(f"  F1-Score difference: {f1_nb_with_stem - f1_nb_no_stem:+.4f}")
print("\nInterpretation: Stemming reduces vocabulary by consolidating word forms")
print("(e.g., 'cares', 'caring', 'cared' -> 'care'), but may reduce interpretability")
print("and lose some sentiment nuances. Effects vary between models.")

### Visualization: Controlled Experiments Summary

In [ ]:
# Create comprehensive comparison visualization (robust to missing configs per model)
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

def _sorted_configs(values):
    """Sort config labels by the first number found (e.g., 5,000 < 10,000), else keep stable order."""
    vals = [v for v in values if pd.notna(v)]
    def key(v):
        m = re.search(r"[\d,]+", str(v))
        return int(m.group(0).replace(",", "")) if m else float("inf")
    # stable sort: numeric first, then original order for ties
    return sorted(vals, key=key)

def _metric_by_config(df, model, metric, configs):
    """Return metric values aligned to configs (NaN if a model/config combo is missing)."""
    s = (df[df["model"] == model]
         .dropna(subset=["config"])
         .drop_duplicates(subset=["config"], keep="last")
         .set_index("config")[metric])
    return s.reindex(configs).to_numpy()

# Experiment 1: Stopwords
exp1_df = pd.DataFrame(exp1_results)
configs1 = _sorted_configs(exp1_df["config"].unique())
x1 = np.arange(len(configs1))
width = 0.2

lr_acc_1 = _metric_by_config(exp1_df, "Logistic Regression", "accuracy", configs1)
lr_f1_1  = _metric_by_config(exp1_df, "Logistic Regression", "f1", configs1)
nb_acc_1 = _metric_by_config(exp1_df, "Naive Bayes", "accuracy", configs1)
nb_f1_1  = _metric_by_config(exp1_df, "Naive Bayes", "f1", configs1)

axes[0].bar(x1 - 1.5*width, lr_acc_1, width, label='LR Accuracy', alpha=0.8, color='steelblue')
axes[0].bar(x1 - 0.5*width, lr_f1_1,  width, label='LR F1-Score', alpha=0.8, color='lightblue')
axes[0].bar(x1 + 0.5*width, nb_acc_1, width, label='NB Accuracy', alpha=0.8, color='coral')
axes[0].bar(x1 + 1.5*width, nb_f1_1,  width, label='NB F1-Score', alpha=0.8, color='lightsalmon')
axes[0].set_xticks(x1)
axes[0].set_xticklabels(['No Domain\nStopwords', 'With Domain\nStopwords'][:len(configs1)], fontsize=9)
axes[0].set_ylabel('Score')
axes[0].set_title('Exp 1: Effect of Domain Stopwords', fontweight='bold')
axes[0].legend(fontsize=8)
axes[0].set_ylim([0.88, 0.98])
axes[0].grid(axis='y', alpha=0.3)

# Experiment 2: Vocabulary Size
# (FIXED: align heights to x2)
exp2_df = pd.DataFrame(exp2_results)
configs2 = _sorted_configs(exp2_df["config"].unique())
x2 = np.arange(len(configs2))

lr_acc_2 = _metric_by_config(exp2_df, "Logistic Regression", "accuracy", configs2)
lr_f1_2  = _metric_by_config(exp2_df, "Logistic Regression", "f1", configs2)
nb_acc_2 = _metric_by_config(exp2_df, "Naive Bayes", "accuracy", configs2)
nb_f1_2  = _metric_by_config(exp2_df, "Naive Bayes", "f1", configs2)

axes[1].bar(x2 - 1.5*width, lr_acc_2, width, label='LR Accuracy', alpha=0.8, color='steelblue')
axes[1].bar(x2 - 0.5*width, lr_f1_2,  width, label='LR F1-Score', alpha=0.8, color='lightblue')
axes[1].bar(x2 + 0.5*width, nb_acc_2, width, label='NB Accuracy', alpha=0.8, color='coral')
axes[1].bar(x2 + 1.5*width, nb_f1_2,  width, label='NB F1-Score', alpha=0.8, color='lightsalmon')
axes[1].set_xticks(x2)

# Prefer your original labels when there are exactly 2 configs; otherwise show config text
if len(configs2) == 2:
    axes[1].set_xticklabels(['5K Features', '10K Features'], fontsize=9)
else:
    axes[1].set_xticklabels([str(c) for c in configs2], fontsize=9, rotation=0)

axes[1].set_ylabel('Score')
axes[1].set_title('Exp 2: Vocabulary Size', fontweight='bold')
axes[1].legend(fontsize=8)
axes[1].set_ylim([0.88, 0.98])
axes[1].grid(axis='y', alpha=0.3)

# Experiment 3: N-grams
exp3_df = pd.DataFrame(exp3_results)
configs3 = _sorted_configs(exp3_df["config"].unique())
x3 = np.arange(len(configs3))

lr_acc_3 = _metric_by_config(exp3_df, "Logistic Regression", "accuracy", configs3)
lr_f1_3  = _metric_by_config(exp3_df, "Logistic Regression", "f1", configs3)
nb_acc_3 = _metric_by_config(exp3_df, "Naive Bayes", "accuracy", configs3)
nb_f1_3  = _metric_by_config(exp3_df, "Naive Bayes", "f1", configs3)

axes[2].bar(x3 - 1.5*width, lr_acc_3, width, label='LR Accuracy', alpha=0.8, color='steelblue')
axes[2].bar(x3 - 0.5*width, lr_f1_3,  width, label='LR F1-Score', alpha=0.8, color='lightblue')
axes[2].bar(x3 + 0.5*width, nb_acc_3, width, label='NB Accuracy', alpha=0.8, color='coral')
axes[2].bar(x3 + 1.5*width, nb_f1_3,  width, label='NB F1-Score', alpha=0.8, color='lightsalmon')
axes[2].set_xticks(x3)
axes[2].set_xticklabels([str(c) for c in configs3], fontsize=9)
axes[2].set_ylabel('Score')
axes[2].set_title('Exp 3: N-gram Features', fontweight='bold')
axes[2].legend(fontsize=8)
axes[2].set_ylim([0.88, 0.98])
axes[2].grid(axis='y', alpha=0.3)

# Experiment 4: Alpha Smoothing
exp4_df = pd.DataFrame(exp4_results)
configs4 = _sorted_configs(exp4_df["config"].unique())
x4 = np.arange(len(configs4))

lr_acc_4 = _metric_by_config(exp4_df, "Logistic Regression", "accuracy", configs4)
lr_f1_4  = _metric_by_config(exp4_df, "Logistic Regression", "f1", configs4)
nb_acc_4 = _metric_by_config(exp4_df, "Naive Bayes", "accuracy", configs4)
nb_f1_4  = _metric_by_config(exp4_df, "Naive Bayes", "f1", configs4)

axes[3].bar(x4 - 1.5*width, lr_acc_4, width, label='LR Accuracy', alpha=0.8, color='steelblue')
axes[3].bar(x4 - 0.5*width, lr_f1_4,  width, label='LR F1-Score', alpha=0.8, color='lightblue')
axes[3].bar(x4 + 0.5*width, nb_acc_4, width, label='NB Accuracy', alpha=0.8, color='coral')
axes[3].bar(x4 + 1.5*width, nb_f1_4,  width, label='NB F1-Score', alpha=0.8, color='lightsalmon')
axes[3].set_xticks(x4)
axes[3].set_xticklabels([str(c) for c in configs4], fontsize=9)
axes[3].set_ylabel('Score')
axes[3].set_title('Exp 4: Naive Bayes Smoothing (alpha)', fontweight='bold')
axes[3].legend(fontsize=8)
axes[3].set_ylim([0.88, 0.98])
axes[3].grid(axis='y', alpha=0.3)

plt.suptitle('Model Performance Comparison Across Experiments', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()
plt.close()
gc.collect()  # Free memory  # Free memory


## Final Model Training: Logistic Regression vs Naive Bayes

Based on experiments, we use the best configuration (10K features, bigrams, domain stopwords) to train and compare two models.

### Finalized Controlled-Experiment Choices

These settings are selected from the controlled experiments and carried into the final model training workflow.

In [ ]:
# Finalized preprocessing and model choices from controlled experiments
finalized_choices = {
    'stopwords': 'English + domain-specific',
    'max_features': 10000,
    'ngram_range': (1, 2),
    'min_df': 5,
    'max_df': 0.8,
    'stemming': False,
    'selected_model': 'LogisticRegression',
    'model_params': {
        'class_weight': 'balanced',
        'C': 1.0,
        'max_iter': 1000,
        'random_state': 42
    }
}

print('Finalized Controlled-Experiment Choices')
for key, value in finalized_choices.items():
    print(f'- {key}: {value}')


### Create Final TF-IDF Features

In [ ]:
# Create TF-IDF features with best configuration
combined_stopwords = list(ENGLISH_STOP_WORDS.union(domain_stopwords))

tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.8,
    stop_words=combined_stopwords,
    token_pattern=r'\b[a-zA-Z]{3,}\b'
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)
X_test_tfidf = tfidf.transform(X_test)

print(f"Final vocabulary size: {len(tfidf.vocabulary_):,} words")
print(f"Feature matrix shape: {X_train_tfidf.shape}")
print(f"Sparsity: {(1 - X_train_tfidf.nnz / (X_train_tfidf.shape[0] * X_train_tfidf.shape[1])):.2%}")

### Model 1: Logistic Regression

**Why Logistic Regression?**
- Provides interpretable coefficients showing each word's contribution to sentiment
- Handles high-dimensional sparse data well
- Outputs probabilities, not just classifications
- `class_weight='balanced'` addresses class imbalance by penalizing errors on minority class more

In [ ]:
# Train Logistic Regression
print("Training Logistic Regression...")
lr = LogisticRegression(
    random_state=42,
    max_iter=1000,
    class_weight='balanced',  # Handles class imbalance
    C=1.0,  # Regularization strength
    solver='liblinear'  # Good for high-dimensional data
)

lr.fit(X_train_tfidf, y_train)
y_test_pred_lr = lr.predict(X_test_tfidf)

# Metrics
acc_lr = accuracy_score(y_test, y_test_pred_lr)
f1_lr = f1_score(y_test, y_test_pred_lr)
prec_lr = precision_score(y_test, y_test_pred_lr)
rec_lr = recall_score(y_test, y_test_pred_lr)

print("LOGISTIC REGRESSION RESULTS")
print(f"Test Accuracy:  {acc_lr:.4f}")
print(f"Precision (Positive): {prec_lr:.4f}")
print(f"Recall (Positive):    {rec_lr:.4f}")
print(f"F1-Score:             {f1_lr:.4f}")
print("\n" + classification_report(y_test, y_test_pred_lr, 
                                   target_names=['Negative', 'Positive']))

### Model 2: Naive Bayes

**Why Naive Bayes?**
- Fast training and prediction (baseline comparison)
- Assumes features are independent given the class (strong but often effective assumption)
- Works well with text data despite independence assumption
- Provides probability estimates based on word frequencies

In [ ]:
# Train Naive Bayes
print("Training Naive Bayes...")
nb = MultinomialNB(
    alpha=1.0  # Laplace smoothing to handle unseen words
)

nb.fit(X_train_tfidf, y_train)
y_test_pred_nb = nb.predict(X_test_tfidf)

# Metrics
acc_nb = accuracy_score(y_test, y_test_pred_nb)
f1_nb = f1_score(y_test, y_test_pred_nb)
prec_nb = precision_score(y_test, y_test_pred_nb)
rec_nb = recall_score(y_test, y_test_pred_nb)

print("NAIVE BAYES RESULTS")
print(f"Test Accuracy:  {acc_nb:.4f}")
print(f"Precision (Positive): {prec_nb:.4f}")
print(f"Recall (Positive):    {rec_nb:.4f}")
print(f"F1-Score:             {f1_nb:.4f}")
print("\n" + classification_report(y_test, y_test_pred_nb, 
                                   target_names=['Negative', 'Positive']))

## Model Comparison: Behavior and Assumptions

In [ ]:
# Create comprehensive comparison
print("MODEL COMPARISON SUMMARY")

comparison_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Logistic Regression': [acc_lr, prec_lr, rec_lr, f1_lr],
    'Naive Bayes': [acc_nb, prec_nb, rec_nb, f1_nb],
    'Difference (LR - NB)': [
        acc_lr - acc_nb,
        prec_lr - prec_nb,
        rec_lr - rec_nb,
        f1_lr - f1_nb
    ]
})

print(comparison_df.to_string(index=False))

print("KEY DIFFERENCES IN MODEL BEHAVIOR")

print("\nLOGISTIC REGRESSION:")
print("  - Models P(positive | words) using logistic function")
print("  - Learns weights showing each word's marginal contribution")
print("  - Handles correlated features (e.g., 'great' and 'amazing' often co-occur)")
print("  - Regularization prevents overfitting on rare words")
print("  - More interpretable: coefficients = log-odds change per word")

print("\nNAIVE BAYES:")
print("  - Assumes features are conditionally independent given class")
print("  - Models P(words | positive) using word frequency distributions")
print("  - Faster training but less flexible than Logistic Regression")
print("  - Can be overconfident due to independence assumption")
print("  - Works surprisingly well despite violated assumptions")

print("\nPERFORMANCE INTERPRETATION:")
if acc_lr > acc_nb:
    print(f"  - Logistic Regression outperforms by {(acc_lr - acc_nb)*100:.2f} percentage points")
    print("  - Likely benefits from modeling word interactions and correlations")
else:
    print(f"  - Naive Bayes outperforms by {(acc_nb - acc_lr)*100:.2f} percentage points")
    print("  - Independence assumption holds reasonably well for this task")

print(f"\n  - Both models achieve >{max(acc_lr, acc_nb):.1%} accuracy")
print("  - Text-based features are highly predictive of sentiment")
print("  - Similar performance suggests sentiment signals are strong and clear")

### Visualization: Model Comparison

In [ ]:
# Create side-by-side comparison visualization
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Metrics comparison
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
lr_scores = [acc_lr, prec_lr, rec_lr, f1_lr]
nb_scores = [acc_nb, prec_nb, rec_nb, f1_nb]

x = np.arange(len(metrics))
width = 0.35

axes[0].bar(x - width/2, lr_scores, width, label='Logistic Regression', alpha=0.8, color='steelblue')
axes[0].bar(x + width/2, nb_scores, width, label='Naive Bayes', alpha=0.8, color='coral')
axes[0].set_ylabel('Score', fontweight='bold')
axes[0].set_title('Performance Metrics Comparison', fontweight='bold', fontsize=12)
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics)
axes[0].legend()
axes[0].set_ylim([0.75, 1.0])
axes[0].grid(axis='y', alpha=0.3)

# Add value labels on bars
for i, (lr_val, nb_val) in enumerate(zip(lr_scores, nb_scores)):
    axes[0].text(i - width/2, lr_val + 0.01, f'{lr_val:.3f}', ha='center', va='bottom', fontsize=8)
    axes[0].text(i + width/2, nb_val + 0.01, f'{nb_val:.3f}', ha='center', va='bottom', fontsize=8)

# Confusion matrices side by side
cm_lr = confusion_matrix(y_test, y_test_pred_lr)
cm_nb = confusion_matrix(y_test, y_test_pred_nb)

# Normalize for percentage display
cm_lr_norm = cm_lr.astype('float') / cm_lr.sum(axis=1)[:, np.newaxis]
cm_nb_norm = cm_nb.astype('float') / cm_nb.sum(axis=1)[:, np.newaxis]

# Create combined heatmap
combined_cm = np.zeros((2, 4))
combined_cm[0, :2] = cm_lr_norm[0, :]
combined_cm[0, 2:] = cm_nb_norm[0, :]
combined_cm[1, :2] = cm_lr_norm[1, :]
combined_cm[1, 2:] = cm_nb_norm[1, :]

sns.heatmap(combined_cm, annot=True, fmt='.3f', cmap='Blues', ax=axes[1],
            xticklabels=['LR: Neg', 'LR: Pos', 'NB: Neg', 'NB: Pos'],
            yticklabels=['True Neg', 'True Pos'],
            cbar_kws={'label': 'Proportion'})
axes[1].set_title('Confusion Matrices (Normalized)', fontweight='bold', fontsize=12)
axes[1].set_xlabel('Predicted Label', fontweight='bold')
axes[1].set_ylabel('True Label', fontweight='bold')

plt.tight_layout()
plt.savefig('../Images/model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()  # Free memory  # Free memory

## Feature Importance and Substantive Interpretation

### Logistic Regression: Top Predictive Words

In [ ]:
# Extract feature importance from Logistic Regression
feature_names = tfidf.get_feature_names_out()
coefficients = lr.coef_[0]

feature_df = pd.DataFrame({
    'word': feature_names,
    'coefficient': coefficients
}).sort_values('coefficient', ascending=False)

print("TOP 20 POSITIVE PREDICTORS (Logistic Regression)")
print(feature_df.head(20)[['word', 'coefficient']].to_string(index=False))

print("TOP 20 NEGATIVE PREDICTORS (Logistic Regression)")
print(feature_df.tail(20)[['word', 'coefficient']].to_string(index=False))

# Save feature importance
feature_df.to_csv('../Assets/feature_importance.csv', index=False)
print("\nFeature importance saved to Assets/feature_importance.csv")

### Naive Bayes: Log-Probability Ratios

In [ ]:
# Extract feature importance from Naive Bayes
# Use log probability ratio: log(P(word|positive) / P(word|negative))
neg_class_log_prob = nb.feature_log_prob_[0]  # P(word | negative)
pos_class_log_prob = nb.feature_log_prob_[1]  # P(word | positive)
log_prob_ratio = pos_class_log_prob - neg_class_log_prob

nb_feature_df = pd.DataFrame({
    'word': feature_names,
    'log_prob_ratio': log_prob_ratio
}).sort_values('log_prob_ratio', ascending=False)

print("TOP 20 POSITIVE PREDICTORS (Naive Bayes)")
print(nb_feature_df.head(20)[['word', 'log_prob_ratio']].to_string(index=False))

print("TOP 20 NEGATIVE PREDICTORS (Naive Bayes)")
print(nb_feature_df.tail(20)[['word', 'log_prob_ratio']].to_string(index=False))

# Save NB feature importance
nb_feature_df.to_csv('../Assets/feature_importance_nb.csv', index=False)
print("\nNaive Bayes feature importance saved to Assets/feature_importance_nb.csv")

### Compare Top Features Between Models

In [ ]:
# Find common top predictors
lr_top_pos = set(feature_df.head(50)['word'])
nb_top_pos = set(nb_feature_df.head(50)['word'])
common_pos = lr_top_pos & nb_top_pos

lr_top_neg = set(feature_df.tail(50)['word'])
nb_top_neg = set(nb_feature_df.tail(50)['word'])
common_neg = lr_top_neg & nb_top_neg

print("AGREEMENT BETWEEN MODELS")
print(f"\nCommon top 50 positive predictors: {len(common_pos)} words")
print("Examples:", sorted(list(common_pos))[:15])

print(f"\nCommon top 50 negative predictors: {len(common_neg)} words")
print("Examples:", sorted(list(common_neg))[:15])

print(f"\nAgreement rate (positive): {len(common_pos)/50:.1%}")
print(f"Agreement rate (negative): {len(common_neg)/50:.1%}")
print("\nInterpretation: High agreement suggests robust, consistent patterns")
print("across different modeling assumptions.")

## Substantive Interpretation: What Do These Results Mean?

### Beyond Performance Metrics

While both models achieve >92% accuracy, the **substantive insights** reveal what students value in their educational experiences:

#### Positive Sentiment Indicators:
1. **Caring and Support**: Words like "cares," "caring," "helped," "helps," "willing" reveal that students highly value instructor approachability and support
2. **Quality of Experience**: "amazing," "awesome," "excellent," "wonderful" indicate strong emotional responses to positive experiences
3. **Ease and Clarity**: "easy," "straightforward," "clear" suggest students appreciate accessible teaching

#### Negative Sentiment Indicators:
1. **Interpersonal Issues**: "rude," "condescending," "unprofessional" are among the strongest negative predictors - **more impactful than difficulty alone**
2. **Organization Problems**: "disorganized," "unorganized," "unclear," "vague" suggest structural course issues
3. **Pedagogical Failures**: "teach" appears as a negative predictor (often in "doesn't teach well"), along with "confusing" and "useless"

#### Key Insights:
- **Interpersonal dynamics matter more than difficulty**: "rude" has a stronger coefficient than "hard"
- **Emotional labor is visible**: "caring" and "willing" show students notice when instructors go beyond baseline requirements
- **Process vs. outcome**: Negative reviews focus on "how" teaching happens (organization, clarity) not just "what" is taught

### Visualization: Top Predictive Words

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Positive words
top_pos = feature_df.head(15)
axes[0].barh(range(len(top_pos)), top_pos['coefficient'], color='green', alpha=0.7)
axes[0].set_yticks(range(len(top_pos)))
axes[0].set_yticklabels(top_pos['word'])
axes[0].set_xlabel('Coefficient (Log-Odds Contribution)', fontweight='bold')
axes[0].set_title('Top 15 Words Predicting POSITIVE Reviews', fontweight='bold')
axes[0].invert_yaxis()
axes[0].grid(axis='x', alpha=0.3)

# Negative words
top_neg = feature_df.tail(15).iloc[::-1]  # Reverse to show most negative first
axes[1].barh(range(len(top_neg)), top_neg['coefficient'], color='red', alpha=0.7)
axes[1].set_yticks(range(len(top_neg)))
axes[1].set_yticklabels(top_neg['word'])
axes[1].set_xlabel('Coefficient (Log-Odds Contribution)', fontweight='bold')
axes[1].set_title('Top 15 Words Predicting NEGATIVE Reviews', fontweight='bold')
axes[1].invert_yaxis()
axes[1].grid(axis='x', alpha=0.3)

plt.suptitle('Logistic Regression: Most Predictive Words for Sentiment', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../Images/actionable_feedback.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()  # Free memory  # Free memory

## Actionable Insights for Instructors

Based on model predictions, here are concrete recommendations:

### What Drives Positive Reviews:
1. **Demonstrate Care**: Be accessible, responsive, and willing to help
2. **Provide Structure**: Students value clarity and organization
3. **Make Content Accessible**: Breaking down complex topics helps

### What Drives Negative Reviews:
1. **Avoid Interpersonal Conflicts**: Professional, respectful communication is critical
2. **Organize Course Materials**: Clear syllabi, consistent grading, structured lectures
3. **Explain Effectively**: Vague or unclear explanations frustrate students

### Limitation of Numerical Ratings Alone:
A 2-star review might stem from:
- Interpersonal issues ("rude," "condescending")
- Poor organization ("disorganized," "unclear")
- Ineffective teaching ("doesn't teach," "confusing")

**Text analysis reveals which specific issue to address**, while the number alone cannot.

## Confusion Matrix Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Logistic Regression Confusion Matrix
cm_lr = confusion_matrix(y_test, y_test_pred_lr)
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues',
           xticklabels=['Negative', 'Positive'],
           yticklabels=['Negative', 'Positive'],
           ax=axes[0])
axes[0].set_title('Logistic Regression', fontweight='bold', fontsize=12)
axes[0].set_ylabel('True Label', fontweight='bold')
axes[0].set_xlabel('Predicted Label', fontweight='bold')

# Naive Bayes Confusion Matrix
cm_nb = confusion_matrix(y_test, y_test_pred_nb)
sns.heatmap(cm_nb, annot=True, fmt='d', cmap='Oranges',
           xticklabels=['Negative', 'Positive'],
           yticklabels=['Negative', 'Positive'],
           ax=axes[1])
axes[1].set_title('Naive Bayes', fontweight='bold', fontsize=12)
axes[1].set_ylabel('True Label', fontweight='bold')
axes[1].set_xlabel('Predicted Label', fontweight='bold')

plt.suptitle('Confusion Matrices - Test Set', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../Images/confusion_matrix_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()  # Free memory  # Free memory

print("Confusion Matrix Analysis:")
print(f"\nLogistic Regression:")
print(f"  True Negatives:  {cm_lr[0,0]:,}")
print(f"  False Positives: {cm_lr[0,1]:,}")
print(f"  False Negatives: {cm_lr[1,0]:,}")
print(f"  True Positives:  {cm_lr[1,1]:,}")

print(f"\nNaive Bayes:")
print(f"  True Negatives:  {cm_nb[0,0]:,}")
print(f"  False Positives: {cm_nb[0,1]:,}")
print(f"  False Negatives: {cm_nb[1,0]:,}")
print(f"  True Positives:  {cm_nb[1,1]:,}")

## Save Models and Results

In [ ]:
import os
os.makedirs('../Assets', exist_ok=True)

# Save models
with open('../Assets/model_lr.pkl', 'wb') as f:
    pickle.dump(lr, f)
print("Saved Logistic Regression model")

with open('../Assets/model_nb.pkl', 'wb') as f:
    pickle.dump(nb, f)
print("Saved Naive Bayes model")

with open('../Assets/vectorizer_tfidf.pkl', 'wb') as f:
    pickle.dump(tfidf, f)
print("Saved TF-IDF vectorizer")

# Save comprehensive results
week4_results = {
    'experiments': {
        'stopwords_comparison': exp1_results,
        'vocabulary_size': exp2_results,
        'ngram_range': exp3_results,
        'stemming': exp4_results
    },
    'model_comparison': {
        'logistic_regression': {
            'accuracy': float(acc_lr),
            'precision': float(prec_lr),
            'recall': float(rec_lr),
            'f1': float(f1_lr)
        },
        'naive_bayes': {
            'accuracy': float(acc_nb),
            'precision': float(prec_nb),
            'recall': float(rec_nb),
            'f1': float(f1_nb)
        }
    },
    'top_features': {
        'top_10_positive': feature_df.head(10)['word'].tolist(),
        'top_10_negative': feature_df.tail(10)['word'].tolist()
    }
}

with open('../Assets/week4_results.json', 'w') as f:
    json.dump(week4_results, f, indent=2)

## Diagnostics, Robustness, and Validity

This section implements diagnostic analyses to assess model reliability, calibration, and potential failure modes.

## Feature Stability Analysis

Feature stability assesses whether model feature importances/coefficients are consistent across different data samples. Unstable features may indicate overfitting or spurious correlations.

**Key Methods:**
1. **K-Fold Feature Importance**: Examine consistency of feature rankings across CV folds
2. **Bootstrap Coefficient Stability**: Measure coefficient variability through bootstrap resampling

### 1. K-Fold Feature Importance

**Purpose**: Evaluate feature importance consistency across different train/test splits.

**Interpretation**:
- **High consistency**: Features are reliably important regardless of data split
- **Low consistency**: Feature importance varies significantly (potential overfitting)
- **Stability score**: Measures how often features appear in top-N across folds

In [ ]:
from sklearn.model_selection import KFold
from collections import defaultdict

print("Performing K-Fold Feature Importance Analysis...")

# Use K-Fold CV to assess feature importance stability
n_folds = 5
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

# Store coefficients from each fold
lr_coef_folds = []
feature_names = tfidf.get_feature_names_out()

# Train model on each fold and extract coefficients
# train_idx and test_idx_fold are the indices for the training and held-out fold sets
for fold_idx, (train_idx, test_idx_fold) in enumerate(kf.split(X_train_tfidf), 1): #Here this 1 means that the counting of fold_idx starts from 1 instead of 0
    print(f"Processing fold {fold_idx}/{n_folds}...")
    
    # Split data for this fold
    X_fold_train = X_train_tfidf[train_idx]
    y_fold_train = y_train.iloc[train_idx]
    
    # Train logistic regression
    lr_fold = LogisticRegression(random_state=42, max_iter=1000, # liblinear is a good choice for small datasets and high-dimensional data, and class_weight='balanced' helps to handle class imbalance by adjusting weights inversely proportional to class frequencies
                                  class_weight='balanced', solver='liblinear') # class weight will help to handle the imbalance in the dataset and solver is set to liblinear which is good for small datasets and high-dimensional data
    lr_fold.fit(X_fold_train, y_fold_train)
    
    # Store coefficients (flatten to 1D array)
    lr_coef_folds.append(lr_fold.coef_.flatten())

# Convert to numpy array for easier manipulation
lr_coef_folds = np.array(lr_coef_folds)  # Shape: (n_folds, n_features)

print(f"\nCoefficient matrix shape: {lr_coef_folds.shape}")
print(f"Analyzing {len(feature_names)} features across {n_folds} folds")

In [ ]:
# Analyze feature stability across folds

# 1. Calculate mean and std of coefficients across folds
coef_mean = lr_coef_folds.mean(axis=0) # axis = 0 means we are calculating the mean for each feature across all folds, calculating in each column the mean of the values in that column
coef_std = lr_coef_folds.std(axis=0)
coef_cv = np.abs(coef_std / (coef_mean + 1e-10))  # Coefficient of variation, 1e-10 added to avoid division by zero

# 2. Calculate stability score for top-K features
# (How often does a feature appear in top-K across folds?)
top_k = 20
stability_scores = np.zeros(len(feature_names))

for fold_coefs in lr_coef_folds:
    # Get indices of top-k features by absolute coefficient value
    top_k_indices = np.argsort(np.abs(fold_coefs))[-top_k:] # [-top_k:] : take the last top_k indices = the largest magnitudes
    stability_scores[top_k_indices] += 1

# Normalize stability scores (0 to 1)
stability_scores = stability_scores / n_folds # 1 means the feature was in top-k for all folds, 0 means it was never in top-k

# 3. Create DataFrame with stability metrics
stability_df = pd.DataFrame({
    'feature': feature_names,
    'coef_mean': coef_mean,
    'coef_std': coef_std,
    'coef_cv': coef_cv,
    'stability_score': stability_scores,
    'abs_coef_mean': np.abs(coef_mean)
})

# Sort by absolute mean coefficient
stability_df = stability_df.sort_values('abs_coef_mean', ascending=False)

# Display top 30 features with stability metrics
print("\nTop 30 Features by Mean Coefficient (with Stability Metrics):")
print(f"{'Feature':<25} {'Mean Coef':>12} {'Std Dev':>12} {'CV':>10} {'Stability':>10}")

for idx, row in stability_df.head(30).iterrows():
    print(f"{row['feature']:<25} {row['coef_mean']:>12.4f} {row['coef_std']:>12.4f} "
          f"{row['coef_cv']:>10.4f} {row['stability_score']:>10.2f}")

print("\nStability Score: Fraction of folds where feature was in top-20")
print("CV (Coefficient of Variation): Std/Mean - lower is more stable")

In [ ]:
# Visualize feature stability across folds

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

# 1. Top features with error bars showing variability
top_15_features = stability_df.head(15)
x_pos = np.arange(len(top_15_features))

axes[0, 0].barh(x_pos, top_15_features['coef_mean'], 
                xerr=top_15_features['coef_std'],
                color='steelblue', alpha=0.7, capsize=5)
axes[0, 0].set_yticks(x_pos)
axes[0, 0].set_yticklabels(top_15_features['feature'], fontsize=9)
axes[0, 0].set_xlabel('Mean Coefficient (±Std Dev)', fontsize=11)
axes[0, 0].set_title('Top 15 Features: Mean Coefficient with Variability', fontsize=12, fontweight='bold')
axes[0, 0].axvline(x=0, color='red', linestyle='--', linewidth=1, alpha=0.5)
axes[0, 0].invert_yaxis()
axes[0, 0].grid(axis='x', alpha=0.3)

# 2. Coefficient stability heatmap for top features
top_20_indices = stability_df.head(20).index
top_20_feature_names = stability_df.head(20)['feature'].values
coef_matrix_top20 = lr_coef_folds[:, top_20_indices].T  # Shape: (20 features, 5 folds)

im = axes[0, 1].imshow(coef_matrix_top20, aspect='auto', cmap='RdBu_r', 
                       vmin=-np.abs(coef_matrix_top20).max(), 
                       vmax=np.abs(coef_matrix_top20).max())
axes[0, 1].set_yticks(np.arange(len(top_20_feature_names)))
axes[0, 1].set_yticklabels(top_20_feature_names, fontsize=9)
axes[0, 1].set_xticks(np.arange(n_folds))
axes[0, 1].set_xticklabels([f'Fold {i+1}' for i in range(n_folds)])
axes[0, 1].set_title('Coefficient Values Across Folds (Top 20 Features)', fontsize=12, fontweight='bold')
plt.colorbar(im, ax=axes[0, 1], label='Coefficient Value')

# 3. Stability score vs mean coefficient
stable_features = stability_df[stability_df['stability_score'] >= 0.6]
unstable_features = stability_df[stability_df['stability_score'] < 0.6]

axes[1, 0].scatter(stable_features['abs_coef_mean'], stable_features['stability_score'], 
                   alpha=0.6, s=50, c='green', label=f'Stable (n={len(stable_features)})')
axes[1, 0].scatter(unstable_features['abs_coef_mean'], unstable_features['stability_score'], 
                   alpha=0.6, s=50, c='red', label=f'Unstable (n={len(unstable_features)})')
axes[1, 0].set_xlabel('Absolute Mean Coefficient', fontsize=11)
axes[1, 0].set_ylabel('Stability Score', fontsize=11)
axes[1, 0].set_title('Feature Stability vs Importance', fontsize=12, fontweight='bold')
axes[1, 0].axhline(y=0.6, color='black', linestyle='--', linewidth=1, alpha=0.5, label='Stability Threshold')
axes[1, 0].legend(fontsize=10)
axes[1, 0].grid(alpha=0.3)

# 4. Coefficient of Variation distribution
axes[1, 1].hist(stability_df['coef_cv'].clip(0, 10), bins=50, color='coral', alpha=0.7, edgecolor='black')
axes[1, 1].axvline(x=stability_df['coef_cv'].median(), color='red', linestyle='--', 
                   linewidth=2, label=f'Median CV = {stability_df["coef_cv"].median():.2f}')
axes[1, 1].set_xlabel('Coefficient of Variation (Std/Mean)', fontsize=11)
axes[1, 1].set_ylabel('Number of Features', fontsize=11)
axes[1, 1].set_title('Distribution of Feature Stability (CV)', fontsize=12, fontweight='bold')
axes[1, 1].legend(fontsize=10)
axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../Images/feature_stability_kfold.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()  # Free memory  # Free memory

print("\nFeature Stability Summary:")
print(f"  - Features with stability score ≥ 0.6: {len(stable_features)} ({len(stable_features)/len(stability_df)*100:.1f}%)")
print(f"  - Features with stability score < 0.6: {len(unstable_features)} ({len(unstable_features)/len(stability_df)*100:.1f}%)")
print(f"  - Median Coefficient of Variation: {stability_df['coef_cv'].median():.4f}")
# The black thing at the top of the graph represents the standard deviation of the coefficient across the folds. The longer the black line, the more variability there is in that feature's importance across different subsets of the data, which indicates lower stability.
# The graph in [1,1] shows how much the coefficient varies across different folds.
# This shows that most of the features are stable whereas there are some features that are extremely 
# unstable with a very high coefficient of variation, which means that their importance is not consistent across different subsets of the data. 
# This could be due to noise or overfitting on specific folds.

### 2. Bootstrap Coefficient Stability

**Purpose**: Assess coefficient robustness through bootstrap resampling (sampling with replacement).

**Interpretation**:
- **Narrow confidence intervals**: Feature coefficients are stable and reliable
- **Wide confidence intervals**: Coefficients vary significantly (less reliable)
- **Sign consistency**: Does the coefficient maintain the same direction (positive/negative)?

In [ ]:
import gc # My computer was crashing when I ran the bootstrap analysis, so I added this to free up memory after the K-Fold analysis before starting the bootstrap analysis.
from sklearn.utils import resample
# Bootstrap (or CV) coefficient stability checks whether the same word keeps the same role across different versions of the dataset. If a word is a strong positive predictor in one fold but becomes a negative predictor in another fold, that would indicate instability.
print("Performing Bootstrap Coefficient Analysis...")
print("This may take a few minutes...\n")

# Bootstrap parameters
n_bootstrap = 30  # Number of bootstrap samples, it means we'll train 30 different logistic regression models
n_samples = X_train_tfidf.shape[0] # n_samples holds a single integer that represents how many training examples exist in the dataset

# Store coefficients from each bootstrap sample
bootstrap_coefs = []

# Perform bootstrap resampling
for i in range(n_bootstrap):
    if (i + 1) % 20 == 0: # This will print the line after this once the iteration hits 20
        print(f"Completed {i + 1}/{n_bootstrap} bootstrap iterations...")
    
    # Resample with replacement
    X_boot, y_boot = resample(X_train_tfidf, y_train, 
                               n_samples=n_samples, 
                               random_state=i, 
                               stratify=y_train)
    
    # Train model on bootstrap sample
    lr_boot = LogisticRegression(random_state=42, max_iter=1000, 
                                  class_weight='balanced', solver='liblinear')
    lr_boot.fit(X_boot, y_boot)
    
    # Store coefficients
    bootstrap_coefs.append(lr_boot.coef_.flatten())

# Convert to numpy array
bootstrap_coefs = np.array(bootstrap_coefs)  # Shape: (n_bootstrap, n_features)

print(f"\nBootstrap coefficient matrix shape: {bootstrap_coefs.shape}")
print(f"Completed {n_bootstrap} bootstrap iterations")

# Free memory
gc.collect()

In [ ]:
import gc
# Analyze bootstrap coefficient stability

# Calculate statistics for each feature
boot_mean = bootstrap_coefs.mean(axis=0)
boot_std = bootstrap_coefs.std(axis=0)
boot_median = np.median(bootstrap_coefs, axis=0)

# Calculate confidence intervals (95%)
# If we repeatedly resample the dataset and retrain the model, 
# then 95% of the coefficient values we get lie between the lower and upper bounds. 
# In this case between 2.5% and 97.5% means it cuts and gives the middle 95% of the distribution of the bootstrap coefficients 
# for each feature. If the confidence interval includes zero, it suggests that the feature's importance is not stable and may not be a reliable predictor.
boot_ci_lower = np.percentile(bootstrap_coefs, 2.5, axis=0)
boot_ci_upper = np.percentile(bootstrap_coefs, 97.5, axis=0)
boot_ci_width = boot_ci_upper - boot_ci_lower

# Calculate sign consistency (what % of bootstrap samples have same sign as median?)
sign_consistency = np.zeros(len(feature_names))
for i in range(len(feature_names)):
    median_sign = np.sign(boot_median[i])
    if median_sign != 0: # It looks at the median coefficient value for one feature and assigns it a label: positive negative or neutral
        same_sign_count = np.sum(np.sign(bootstrap_coefs[:, i]) == median_sign) # Counts how many bootstrap runs have the same sign as the median.
        sign_consistency[i] = same_sign_count / n_bootstrap
    else:
        sign_consistency[i] = 0.5  # Neutral for features with median near zero

# Check if confidence interval includes zero (unstable sign)
includes_zero = (boot_ci_lower * boot_ci_upper) <= 0
# includes_zero[i] = True means the feature’s effect might be positive or negative 
# sign unstable / not reliable False means it stays confidently on one side of zero.
# Create DataFrame with bootstrap statistics
bootstrap_df = pd.DataFrame({
    'feature': feature_names,
    'boot_mean': boot_mean,
    'boot_median': boot_median,
    'boot_std': boot_std,
    'ci_lower': boot_ci_lower,
    'ci_upper': boot_ci_upper,
    'ci_width': boot_ci_width,
    'sign_consistency': sign_consistency,
    'includes_zero': includes_zero,
    'abs_boot_median': np.abs(boot_median)
})

# Sort by absolute median coefficient
bootstrap_df = bootstrap_df.sort_values('abs_boot_median', ascending=False)

# Display top 30 features with bootstrap statistics
print("\nTop 30 Features by Bootstrap Median Coefficient:")
print(f"{'Feature':<25} {'Median':>10} {'Std':>10} {'CI Width':>10} {'Sign Cons':>10} {'Zero in CI':>12}")

for idx, row in bootstrap_df.head(30).iterrows():
    zero_marker = "Yes" if row['includes_zero'] else "No"
    print(f"{row['feature']:<25} {row['boot_median']:>10.4f} {row['boot_std']:>10.4f} "
          f"{row['ci_width']:>10.4f} {row['sign_consistency']:>10.2f} {zero_marker:>12}")

print("\nSign Consistency: Fraction of bootstrap samples with same sign as median")
print("Zero in CI: Whether 95% CI includes zero (sign instability)")

# Free memory
gc.collect()

In [ ]:
import gc
# Visualize bootstrap coefficient stability

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

# 1. Top features with 95% confidence intervals
top_20_boot = bootstrap_df.head(20)
x_pos = np.arange(len(top_20_boot))

# Create error bars from CI
yerr_lower = top_20_boot['boot_median'] - top_20_boot['ci_lower']
yerr_upper = top_20_boot['ci_upper'] - top_20_boot['boot_median']

axes[0, 0].barh(x_pos, top_20_boot['boot_median'], 
                xerr=[yerr_lower, yerr_upper],
                color='steelblue', alpha=0.7, capsize=4)
axes[0, 0].set_yticks(x_pos)
axes[0, 0].set_yticklabels(top_20_boot['feature'], fontsize=9)
axes[0, 0].set_xlabel('Bootstrap Median Coefficient (95% CI)', fontsize=11)
axes[0, 0].set_title('Top 20 Features: Bootstrap Coefficients with 95% CI', 
                     fontsize=12, fontweight='bold')
axes[0, 0].axvline(x=0, color='red', linestyle='--', linewidth=1, alpha=0.5)
axes[0, 0].invert_yaxis()
axes[0, 0].grid(axis='x', alpha=0.3)

# 2. Distribution of coefficients for top 6 features (violin plot)
top_6_indices = bootstrap_df.head(6).index
top_6_names = bootstrap_df.head(6)['feature'].values
top_6_coefs = bootstrap_coefs[:, top_6_indices]

# Create violin plot
parts = axes[0, 1].violinplot([top_6_coefs[:, i] for i in range(6)],
                               positions=range(6),
                               showmeans=True, showmedians=True)

# Color the violins
for pc in parts['bodies']:
    pc.set_facecolor('lightblue')
    pc.set_alpha(0.7)

axes[0, 1].set_xticks(range(6))
axes[0, 1].set_xticklabels(top_6_names, rotation=45, ha='right', fontsize=9)
axes[0, 1].set_ylabel('Coefficient Value', fontsize=11)
axes[0, 1].set_title('Bootstrap Distribution of Top 6 Features', 
                     fontsize=12, fontweight='bold')
axes[0, 1].axhline(y=0, color='red', linestyle='--', linewidth=1, alpha=0.5)
axes[0, 1].grid(axis='y', alpha=0.3)

# 3. CI Width vs Coefficient Magnitude
stable_boot = bootstrap_df[~bootstrap_df['includes_zero']]
unstable_boot = bootstrap_df[bootstrap_df['includes_zero']]

axes[1, 0].scatter(stable_boot['abs_boot_median'], stable_boot['ci_width'],
                   alpha=0.5, s=30, c='green', label=f'Stable Sign (n={len(stable_boot)})')
axes[1, 0].scatter(unstable_boot['abs_boot_median'], unstable_boot['ci_width'],
                   alpha=0.5, s=30, c='red', label=f'Unstable Sign (n={len(unstable_boot)})')
axes[1, 0].set_xlabel('Absolute Median Coefficient', fontsize=11)
axes[1, 0].set_ylabel('95% CI Width', fontsize=11)
axes[1, 0].set_title('Coefficient Stability: CI Width vs Magnitude', 
                     fontsize=12, fontweight='bold')
axes[1, 0].legend(fontsize=10)
axes[1, 0].grid(alpha=0.3)

# 4. Sign consistency distribution
axes[1, 1].hist(bootstrap_df['sign_consistency'], bins=50, 
                color='purple', alpha=0.7, edgecolor='black')
axes[1, 1].axvline(x=0.95, color='green', linestyle='--', linewidth=2, 
                   label='95% Threshold')
axes[1, 1].axvline(x=bootstrap_df['sign_consistency'].median(), color='red', 
                   linestyle='--', linewidth=2, 
                   label=f'Median = {bootstrap_df["sign_consistency"].median():.2f}')
axes[1, 1].set_xlabel('Sign Consistency (fraction)', fontsize=11)
axes[1, 1].set_ylabel('Number of Features', fontsize=11)
axes[1, 1].set_title('Distribution of Sign Consistency Across Features', 
                     fontsize=12, fontweight='bold')
axes[1, 1].legend(fontsize=10)
axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../Images/feature_stability_bootstrap.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()  # Free memory

# Summary statistics
n_stable_sign = len(stable_boot)
n_unstable_sign = len(unstable_boot)
high_sign_consistency = len(bootstrap_df[bootstrap_df['sign_consistency'] >= 0.95])

print("\nBootstrap Stability Summary:")
print(f"  - Features with stable sign (CI excludes 0): {n_stable_sign} ({n_stable_sign/len(bootstrap_df)*100:.1f}%)")
print(f"  - Features with unstable sign (CI includes 0): {n_unstable_sign} ({n_unstable_sign/len(bootstrap_df)*100:.1f}%)")
print(f"  - Features with sign consistency ≥ 95%: {high_sign_consistency} ({high_sign_consistency/len(bootstrap_df)*100:.1f}%)")
print(f"  - Median CI width: {bootstrap_df['ci_width'].median():.4f}")

# Free memory
gc.collect()
# The black mark is drawn using the 2.5th and 97.5th percentiles of the feature’s 
# bootstrap coefficient values, enclosing the middle 95% of observed variability.

# The middle dark blue line represents the median coefficient, the upper dark blue line represents the 97.5th percentile, 
# and the lower dark blue line represents the 2.5th percentile of the bootstrap coefficient distribution.

# The y-axis represents the width of the 95% bootstrap confidence interval, 
# computed as the difference between the 97.5th and 2.5th percentile coefficient values for each feature.

# Sign consistency is the fraction of bootstrap-trained models in which a feature’s coefficient has the same sign as its median coefficient.

### Feature Stability Analysis Summary

| Method | Key Metric | What It Measures | Interpretation |
|--------|-----------|------------------|----------------|
| **K-Fold CV** | Stability Score | How often a feature appears in top-K across folds | Higher = more consistently important |
| **K-Fold CV** | Coefficient of Variation | Std/Mean of coefficients across folds | Lower = more stable |
| **Bootstrap** | 95% CI Width | Variability of coefficient estimates | Narrower = more reliable |
| **Bootstrap** | Sign Consistency | % of samples with same sign | Higher = more confident in direction |
| **Bootstrap** | CI Includes Zero | Whether effect direction is uncertain | No = stable, Yes = unstable |

**Key Findings:**
- **Stable features** show high stability scores (≥0.6), low CV, narrow confidence intervals, and high sign consistency (≥95%)
- **Unstable features** may be overfitting to noise or sensitive to specific data samples
- Features with CI including zero have uncertain directional effects and should be interpreted cautiously

In [ ]:
# Compare and identify most stable features across both methods

# Merge the two dataframes
combined_stability = stability_df.merge(
    bootstrap_df[['feature', 'boot_median', 'ci_width', 'sign_consistency', 'includes_zero']], 
    on='feature', 
    how='inner'
)
# on='feature', Tells pandas to join the two tables by matching the feature column in both. how='inner' ) 
# how='inner' means: keep only features that exist in both dataframes.

# Define stability criteria
# High stability = high k-fold stability score, low CV, narrow CI, high sign consistency
combined_stability['is_stable'] = (
    (combined_stability['stability_score'] >= 0.6) &
    (combined_stability['sign_consistency'] >= 0.95) &
    (~combined_stability['includes_zero'])
)

# Sort by stability score
stable_features_combined = combined_stability[combined_stability['is_stable']].sort_values(
    'abs_coef_mean', ascending=False
)

unstable_features_combined = combined_stability[~combined_stability['is_stable']].sort_values(
    'abs_coef_mean', ascending=False
)

print("MOST STABLE FEATURES (Pass all stability criteria)")
print(f"Found {len(stable_features_combined)} highly stable features\n")
print(f"{'Feature':<25} {'Mean Coef':>12} {'K-Fold Stab':>12} {'Sign Cons':>12} {'CI Width':>12}")

for idx, row in stable_features_combined.head(20).iterrows():
    print(f"{row['feature']:<25} {row['coef_mean']:>12.4f} {row['stability_score']:>12.2f} "
          f"{row['sign_consistency']:>12.2f} {row['ci_width']:>12.4f}")

print("POTENTIALLY UNSTABLE FEATURES (Fail one or more stability criteria)")
print(f"Found {len(unstable_features_combined)} features with stability concerns\n")
print(f"{'Feature':<25} {'Mean Coef':>12} {'K-Fold Stab':>12} {'Sign Cons':>12} {'Zero in CI':>12}")

for idx, row in unstable_features_combined.head(20).iterrows():
    zero_marker = "Yes" if row['includes_zero'] else "No"
    print(f"{row['feature']:<25} {row['coef_mean']:>12.4f} {row['stability_score']:>12.2f} "
          f"{row['sign_consistency']:>12.2f} {zero_marker:>12}")

print(f"Overall: {len(stable_features_combined)}/{len(combined_stability)} features "
      f"({len(stable_features_combined)/len(combined_stability)*100:.1f}%) are highly stable")

# The upper table lists features whose coefficients are consistently strong and directionally stable across both K-fold cross-validation 
# and bootstrap resampling, indicating robust and reliable predictors whose importance does not depend on the specific training split.

# The lower table contains features that, despite often having large coefficients and stable signs, do not consistently rank among the 
# most important features across K-fold splits, suggesting their influence is context-dependent or interchangeable with correlated features 
# rather than universally dominant.

In [ ]:
# Create comprehensive stability comparison visualization

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Coefficient of Variation vs CI Width
axes[0].scatter(combined_stability[combined_stability['is_stable']]['coef_cv'].clip(0, 5),
                combined_stability[combined_stability['is_stable']]['ci_width'],
                alpha=0.6, s=60, c='green', label='Stable')
axes[0].scatter(combined_stability[~combined_stability['is_stable']]['coef_cv'].clip(0, 5),
                combined_stability[~combined_stability['is_stable']]['ci_width'],
                alpha=0.6, s=60, c='red', label='Unstable')
axes[0].set_xlabel('K-Fold Coefficient of Variation (capped at 5)', fontsize=11)
axes[0].set_ylabel('Bootstrap 95% CI Width', fontsize=11)
axes[0].set_title('Comparing Variability Metrics', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)

# 2. Top 15 stable features comparison
top_15_stable = stable_features_combined.head(15)
x = np.arange(len(top_15_stable))
width = 0.35

axes[1].barh(x - width/2, top_15_stable['stability_score'], width, 
             label='K-Fold Stability', color='steelblue', alpha=0.8)
axes[1].barh(x + width/2, top_15_stable['sign_consistency'], width,
             label='Bootstrap Sign Cons', color='coral', alpha=0.8)
axes[1].set_yticks(x)
axes[1].set_yticklabels(top_15_stable['feature'], fontsize=9)
axes[1].set_xlabel('Score', fontsize=11)
axes[1].set_title('Top 15 Stable Features: Dual Stability Metrics', 
                  fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].invert_yaxis()
axes[1].grid(axis='x', alpha=0.3)

# 3. Stability classification pie chart
stability_categories = combined_stability.groupby('is_stable').size()
colors = ['#ff6b6b', '#51cf66']
labels = [f'Unstable\n({unstable_features_combined.shape[0]})', 
          f'Stable\n({stable_features_combined.shape[0]})']

wedges, texts, autotexts = axes[2].pie(stability_categories, 
                                       labels=labels,
                                       colors=colors,
                                       autopct='%1.1f%%',
                                       startangle=90,
                                       textprops={'fontsize': 12, 'weight': 'bold'})
axes[2].set_title('Overall Feature Stability Classification', 
                  fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('../Images/feature_stability_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()  # Free memory  # Free memory

print("Stability analysis complete!")
print(f"Saved visualizations:")
print(f"  - ../Images/feature_stability_kfold.png")
print(f"  - ../Images/feature_stability_bootstrap.png")
print(f"  - ../Images/feature_stability_comparison.png")

# The K-fold coefficient of variation is the ratio of the standard deviation to the mean of a feature’s coefficients across 
# cross-validation folds, quantifying how stable the feature’s magnitude is relative to its average effect.

## Robustness Checks

Robustness checks assess whether model performance is consistent across different conditions and not dependent on specific data characteristics or sampling choices.

**Key Methods:**
1. **Cross-Validation**: Systematic evaluation across multiple train/test splits
2. **Row-Order Validation**: Performance stability across different time periods
3. **Sample Size Sensitivity**: How performance varies with training data size

### 1. Cross-Validation Robustness

**Purpose**: Evaluate model performance across multiple train/test splits to ensure results are not dependent on a single data partition.

**Approach**: 
- Use stratified k-fold cross-validation to maintain class balance
- Compute multiple performance metrics across folds
- Assess variability in performance

In [ ]:
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

print("Performing Cross-Validation Robustness Check...")

# Define scoring metrics
scoring = {
    'accuracy': make_scorer(accuracy_score),
    'precision': make_scorer(precision_score, average='weighted', zero_division=0),
    'recall': make_scorer(recall_score, average='weighted', zero_division=0),
    'f1': make_scorer(f1_score, average='weighted', zero_division=0),
    'roc_auc': make_scorer(roc_auc_score, needs_proba=True)
}

# Setup stratified k-fold
n_folds_cv = 5
cv_splitter = StratifiedKFold(n_splits=n_folds_cv, shuffle=True, random_state=42)

# Perform cross-validation for Logistic Regression
print("\n1. Logistic Regression Cross-Validation...")
lr_cv_model = LogisticRegression(random_state=42, max_iter=1000, 
                                  class_weight='balanced', solver='liblinear')

lr_cv_results = cross_validate(
    lr_cv_model, 
    X_train_tfidf, 
    y_train,
    cv=cv_splitter,
    scoring=scoring,
    return_train_score=True,
    n_jobs=2, 
    verbose=1
)

print("Logistic Regression completed.")

# Free memory before next model
gc.collect()

# Perform cross-validation for Naive Bayes
print("\n2. Naive Bayes Cross-Validation...")
nb_cv_model = MultinomialNB()

# Use sparse matrix directly - NO conversion to dense!
nb_cv_results = cross_validate(
    nb_cv_model,
    X_train_tfidf,  # Using sparse matrix directly instead of X_train_dense
    y_train,
    cv=cv_splitter,
    scoring=scoring,
    return_train_score=True,
    n_jobs=2,  # Changed from -1 to reduce memory usage
    verbose=1
)

print("Naive Bayes completed.")
print("Cross-validation complete!")


In [ ]:
# Analyze cross-validation results

# Extract metrics for analysis
metrics = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

# Create summary DataFrames
cv_summary_data = []

for metric in metrics:
    # Test scores
    lr_test_mean = lr_cv_results[f'test_{metric}'].mean()
    lr_test_std = lr_cv_results[f'test_{metric}'].std()
    nb_test_mean = nb_cv_results[f'test_{metric}'].mean()
    nb_test_std = nb_cv_results[f'test_{metric}'].std()
    
    # Train scores
    lr_train_mean = lr_cv_results[f'train_{metric}'].mean()
    lr_train_std = lr_cv_results[f'train_{metric}'].std()
    nb_train_mean = nb_cv_results[f'train_{metric}'].mean()
    nb_train_std = nb_cv_results[f'train_{metric}'].std()
    
    cv_summary_data.append({
        'Metric': metric.upper(),
        'LR_Test_Mean': lr_test_mean,
        'LR_Test_Std': lr_test_std,
        'LR_Train_Mean': lr_train_mean,
        'NB_Test_Mean': nb_test_mean,
        'NB_Test_Std': nb_test_std,
        'NB_Train_Mean': nb_train_mean
    })

cv_summary_df = pd.DataFrame(cv_summary_data)

# Display results
print("Cross-Validation Results Summary (5-Fold)")
print(f"{'Metric':<12} {'LR Test':>18} {'LR Train':>12} {'NB Test':>18} {'NB Train':>12}")

for _, row in cv_summary_df.iterrows():
    print(f"{row['Metric']:<12} "
          f"{row['LR_Test_Mean']:>8.4f} ± {row['LR_Test_Std']:>6.4f}  "
          f"{row['LR_Train_Mean']:>10.4f}  "
          f"{row['NB_Test_Mean']:>8.4f} ± {row['NB_Test_Std']:>6.4f}  "
          f"{row['NB_Train_Mean']:>10.4f}")


# Calculate coefficient of variation for each metric (test scores)
print("\nCoefficient of Variation (CV) - Test Scores:")
print(f"{'Metric':<12} {'LR CV':>12} {'NB CV':>12} {'Interpretation':>30}")

for _, row in cv_summary_df.iterrows():
    lr_cv = (row['LR_Test_Std'] / row['LR_Test_Mean']) * 100
    nb_cv = (row['NB_Test_Std'] / row['NB_Test_Mean']) * 100
    interpretation = "Highly Stable" if max(lr_cv, nb_cv) < 5 else ("Stable" if max(lr_cv, nb_cv) < 10 else "Variable")
    print(f"{row['Metric']:<12} {lr_cv:>10.2f}%  {nb_cv:>10.2f}%  {interpretation:>30}")

print("\nLower CV% indicates more stable/robust performance across folds")


In [ ]:
# Visualize cross-validation results

fig, axes = plt.subplots(2, 3, figsize=(14, 10))
axes = axes.flatten()

# Plot each metric
for idx, metric in enumerate(metrics):
    ax = axes[idx]
    
    # Get fold-wise scores
    lr_test_scores = lr_cv_results[f'test_{metric}']
    lr_train_scores = lr_cv_results[f'train_{metric}']
    nb_test_scores = nb_cv_results[f'test_{metric}']
    nb_train_scores = nb_cv_results[f'train_{metric}']
    
    x = np.arange(n_folds_cv)
    width = 0.2
    
    # Plot bars
    ax.bar(x - 1.5*width, lr_train_scores, width, label='LR Train', 
           color='steelblue', alpha=0.7)
    ax.bar(x - 0.5*width, lr_test_scores, width, label='LR Test', 
           color='steelblue', alpha=1.0, edgecolor='black', linewidth=1)
    ax.bar(x + 0.5*width, nb_train_scores, width, label='NB Train', 
           color='coral', alpha=0.7)
    ax.bar(x + 1.5*width, nb_test_scores, width, label='NB Test', 
           color='coral', alpha=1.0, edgecolor='black', linewidth=1)
    
    # Add mean lines
    ax.axhline(y=lr_test_scores.mean(), color='steelblue', linestyle='--', 
               linewidth=2, alpha=0.8, label=f'LR Mean: {lr_test_scores.mean():.3f}')
    ax.axhline(y=nb_test_scores.mean(), color='coral', linestyle='--', 
               linewidth=2, alpha=0.8, label=f'NB Mean: {nb_test_scores.mean():.3f}')
    
    ax.set_xlabel('Fold', fontsize=10)
    ax.set_ylabel(metric.upper(), fontsize=10)
    ax.set_title(f'{metric.upper()} Across Folds', fontsize=11, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels([f'{i+1}' for i in range(n_folds_cv)])
    ax.legend(fontsize=8, loc='lower right')
    ax.grid(axis='y', alpha=0.3)

# Remove the 6th subplot (we only have 5 metrics)
fig.delaxes(axes[5])

plt.tight_layout()
plt.savefig('../Images/robustness_cross_validation.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()  # Free memory  # Free memory

print("\nVisualization saved: ../Images/robustness_cross_validation.png")


### 2. Row-Order Validation

**Purpose**: Assess whether models maintain performance when trained on older data and tested on newer data (sequential row-order generalization simulation).

**Ideal Approach**: Split data by time periods (e.g., train on 2003-2020, test on 2021-2024)

**Current Dataset**: Does not contain temporal (date/year) information.

**Alternative Approach**: Simulate row-order split using sequential data ordering to mimic sequential deployment-order scenarios.

In [ ]:
print("Performing Row-Order Validation (Sequential Split Simulation)...")

# Since we don't have temporal data, we'll use sequential splits to simulate row-order validation
# This assumes earlier rows represent older data and later rows represent newer data

# Define row-order splits (simulating different time periods)
# We'll create 3 periods: Early (60%), Middle (20%), Recent (20%)
n_samples = len(y_train)
split_early = int(0.60 * n_samples)  # "Historical" data (2003-2015 simulation)
split_middle = int(0.80 * n_samples)  # "Mid-period" data (2016-2020 simulation)

# Create row-order splits
X_early = X_train_tfidf[:split_early]
y_early = y_train.iloc[:split_early]

X_middle = X_train_tfidf[split_early:split_middle]
y_middle = y_train.iloc[split_early:split_middle]

X_recent = X_train_tfidf[split_middle:]
y_recent = y_train.iloc[split_middle:]

print(f"\nRow-Order Split Sizes:")
print(f"  Early period (60%):  {len(y_early):,} samples")
print(f"  Middle period (20%): {len(y_middle):,} samples")
print(f"  Recent period (20%): {len(y_recent):,} samples")

# Store row-order validation results
row_order_results = []

# Scenario 1: Train on Early, Test on Middle & Recent
print("\n1. Training on Early period, testing on Middle and Recent periods...")
lr_temp1 = LogisticRegression(random_state=42, max_iter=1000, 
                                class_weight='balanced', solver='liblinear')
nb_temp1 = MultinomialNB()

lr_temp1.fit(X_early, y_early)
nb_temp1.fit(X_early.toarray(), y_early)

# Test on middle period
lr_acc_middle = accuracy_score(y_middle, lr_temp1.predict(X_middle))
lr_f1_middle = f1_score(y_middle, lr_temp1.predict(X_middle), average='weighted')
nb_acc_middle = accuracy_score(y_middle, nb_temp1.predict(X_middle.toarray()))
nb_f1_middle = f1_score(y_middle, nb_temp1.predict(X_middle.toarray()), average='weighted')

# Test on recent period
lr_acc_recent = accuracy_score(y_recent, lr_temp1.predict(X_recent))
lr_f1_recent = f1_score(y_recent, lr_temp1.predict(X_recent), average='weighted')
nb_acc_recent = accuracy_score(y_recent, nb_temp1.predict(X_recent.toarray()))
nb_f1_recent = f1_score(y_recent, nb_temp1.predict(X_recent.toarray()), average='weighted')

row_order_results.append({
    'Train Period': 'Early (60%)',
    'Test Period': 'Middle (20%)',
    'LR_Acc': lr_acc_middle,
    'LR_F1': lr_f1_middle,
    'NB_Acc': nb_acc_middle,
    'NB_F1': nb_f1_middle
})

row_order_results.append({
    'Train Period': 'Early (60%)',
    'Test Period': 'Recent (20%)',
    'LR_Acc': lr_acc_recent,
    'LR_F1': lr_f1_recent,
    'NB_Acc': nb_acc_recent,
    'NB_F1': nb_f1_recent
})

# Scenario 2: Train on Early+Middle, Test on Recent
print("2. Training on Early+Middle periods, testing on Recent period...")
X_early_middle = X_train_tfidf[:split_middle]
y_early_middle = y_train.iloc[:split_middle]

lr_temp2 = LogisticRegression(random_state=42, max_iter=1000, 
                                class_weight='balanced', solver='liblinear')
nb_temp2 = MultinomialNB()

lr_temp2.fit(X_early_middle, y_early_middle)
nb_temp2.fit(X_early_middle.toarray(), y_early_middle)

lr_acc_recent2 = accuracy_score(y_recent, lr_temp2.predict(X_recent))
lr_f1_recent2 = f1_score(y_recent, lr_temp2.predict(X_recent), average='weighted')
nb_acc_recent2 = accuracy_score(y_recent, nb_temp2.predict(X_recent.toarray()))
nb_f1_recent2 = f1_score(y_recent, nb_temp2.predict(X_recent.toarray()), average='weighted')

row_order_results.append({
    'Train Period': 'Early+Middle (80%)',
    'Test Period': 'Recent (20%)',
    'LR_Acc': lr_acc_recent2,
    'LR_F1': lr_f1_recent2,
    'NB_Acc': nb_acc_recent2,
    'NB_F1': nb_f1_recent2
})

# Compare with standard train/test (for reference)
lr_acc_test = accuracy_score(y_test, lr.predict(X_test_tfidf))
lr_f1_test = f1_score(y_test, lr.predict(X_test_tfidf), average='weighted')
nb_acc_test = accuracy_score(y_test, nb.predict(X_test_tfidf.toarray()))
nb_f1_test = f1_score(y_test, nb.predict(X_test_tfidf.toarray()), average='weighted')

row_order_results.append({
    'Train Period': 'Standard Train',
    'Test Period': 'Standard Test',
    'LR_Acc': lr_acc_test,
    'LR_F1': lr_f1_test,
    'NB_Acc': nb_acc_test,
    'NB_F1': nb_f1_test
})

print("\nSequential row-order validation complete!")


In [ ]:
# Display row-order validation results

row_order_df = pd.DataFrame(row_order_results)

print("\nRow-Order Validation Results")
print(f"{'Train Period':<25} {'Test Period':<20} {'LR Acc':>12} {'LR F1':>12} {'NB Acc':>12} {'NB F1':>12}")

for _, row in row_order_df.iterrows():
    print(f"{row['Train Period']:<25} {row['Test Period']:<20} "
          f"{row['LR_Acc']:>12.4f} {row['LR_F1']:>12.4f} "
          f"{row['NB_Acc']:>12.4f} {row['NB_F1']:>12.4f}")


# Calculate performance degradation from standard to row-order
print("\n Row-Order Generalization Analysis:")

standard_lr_acc = row_order_df[row_order_df['Train Period'] == 'Standard Train']['LR_Acc'].values[0]
standard_lr_f1 = row_order_df[row_order_df['Train Period'] == 'Standard Train']['LR_F1'].values[0]
standard_nb_acc = row_order_df[row_order_df['Train Period'] == 'Standard Train']['NB_Acc'].values[0]
standard_nb_f1 = row_order_df[row_order_df['Train Period'] == 'Standard Train']['NB_F1'].values[0]

# Compare Early->Recent performance
early_to_recent = row_order_df[(row_order_df['Train Period'] == 'Early (60%)') & 
                               (row_order_df['Test Period'] == 'Recent (20%)')].iloc[0]

print(f"\nStandard Testidation Performance (Baseline):")
print(f"  LR: Accuracy={standard_lr_acc:.4f}, F1={standard_lr_f1:.4f}")
print(f"  NB: Accuracy={standard_nb_acc:.4f}, F1={standard_nb_f1:.4f}")

print(f"\nRow-Order Validation Performance (Early -> Recent):")
print(f"  LR: Accuracy={early_to_recent['LR_Acc']:.4f}, F1={early_to_recent['LR_F1']:.4f}")
print(f"  NB: Accuracy={early_to_recent['NB_Acc']:.4f}, F1={early_to_recent['NB_F1']:.4f}")

lr_acc_drop = ((standard_lr_acc - early_to_recent['LR_Acc']) / standard_lr_acc) * 100
lr_f1_drop = ((standard_lr_f1 - early_to_recent['LR_F1']) / standard_lr_f1) * 100
nb_acc_drop = ((standard_nb_acc - early_to_recent['NB_Acc']) / standard_nb_acc) * 100
nb_f1_drop = ((standard_nb_f1 - early_to_recent['NB_F1']) / standard_nb_f1) * 100

print(f"\nPerformance Change (Standard vs Row-Order):")
print(f"  LR: Accuracy {lr_acc_drop:+.2f}%, F1 {lr_f1_drop:+.2f}%")
print(f"  NB: Accuracy {nb_acc_drop:+.2f}%, F1 {nb_f1_drop:+.2f}%")

if abs(lr_acc_drop) < 5 and abs(nb_acc_drop) < 5:
    print("\nModels show good row-order robustness (< 5% performance change)")
elif abs(lr_acc_drop) < 10 and abs(nb_acc_drop) < 10:
    print("\nModels show moderate row-order robustness (5-10% performance change)")
else:
    print("\nModels show poor row-order robustness (> 10% performance change)")


In [ ]:
# Visualize row-order validation results

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Prepare data for plotting
scenarios = ['Early→Middle', 'Early→Recent', 'Early+Mid→Recent', 'Standard']
lr_acc_scores = [
    row_order_df.iloc[0]['LR_Acc'],
    row_order_df.iloc[1]['LR_Acc'],
    row_order_df.iloc[2]['LR_Acc'],
    row_order_df.iloc[3]['LR_Acc']
]
lr_f1_scores = [
    row_order_df.iloc[0]['LR_F1'],
    row_order_df.iloc[1]['LR_F1'],
    row_order_df.iloc[2]['LR_F1'],
    row_order_df.iloc[3]['LR_F1']
]
nb_acc_scores = [
    row_order_df.iloc[0]['NB_Acc'],
    row_order_df.iloc[1]['NB_Acc'],
    row_order_df.iloc[2]['NB_Acc'],
    row_order_df.iloc[3]['NB_Acc']
]
nb_f1_scores = [
    row_order_df.iloc[0]['NB_F1'],
    row_order_df.iloc[1]['NB_F1'],
    row_order_df.iloc[2]['NB_F1'],
    row_order_df.iloc[3]['NB_F1']
]

x = np.arange(len(scenarios))
width = 0.35

# Plot 1: Accuracy comparison
axes[0].bar(x - width/2, lr_acc_scores, width, label='Logistic Regression', 
            color='steelblue', alpha=0.8)
axes[0].bar(x + width/2, nb_acc_scores, width, label='Naive Bayes', 
            color='coral', alpha=0.8)
axes[0].set_xlabel('Train → Test Scenario', fontsize=11)
axes[0].set_ylabel('Accuracy', fontsize=11)
axes[0].set_title('Row-Order Validation: Accuracy Comparison', fontsize=12, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(scenarios, rotation=15, ha='right')
axes[0].legend(fontsize=10)
axes[0].grid(axis='y', alpha=0.3)
axes[0].set_ylim([min(min(lr_acc_scores), min(nb_acc_scores)) - 0.05, 1.0])

# Add value labels on bars
for i, (lr_val, nb_val) in enumerate(zip(lr_acc_scores, nb_acc_scores)):
    axes[0].text(i - width/2, lr_val + 0.01, f'{lr_val:.3f}', 
                 ha='center', va='bottom', fontsize=9)
    axes[0].text(i + width/2, nb_val + 0.01, f'{nb_val:.3f}', 
                 ha='center', va='bottom', fontsize=9)

# Plot 2: F1 Score comparison
axes[1].bar(x - width/2, lr_f1_scores, width, label='Logistic Regression', 
            color='steelblue', alpha=0.8)
axes[1].bar(x + width/2, nb_f1_scores, width, label='Naive Bayes', 
            color='coral', alpha=0.8)
axes[1].set_xlabel('Train → Test Scenario', fontsize=11)
axes[1].set_ylabel('F1 Score', fontsize=11)
axes[1].set_title('Row-Order Validation: F1 Score Comparison', fontsize=12, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(scenarios, rotation=15, ha='right')
axes[1].legend(fontsize=10)
axes[1].grid(axis='y', alpha=0.3)
axes[1].set_ylim([min(min(lr_f1_scores), min(nb_f1_scores)) - 0.05, 1.0])

# Add value labels on bars
for i, (lr_val, nb_val) in enumerate(zip(lr_f1_scores, nb_f1_scores)):
    axes[1].text(i - width/2, lr_val + 0.01, f'{lr_val:.3f}', 
                 ha='center', va='bottom', fontsize=9)
    axes[1].text(i + width/2, nb_val + 0.01, f'{nb_val:.3f}', 
                 ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('../Images/robustness_row_order_validation.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()  # Free memory  # Free memory

print("\nVisualization saved: ../Images/robustness_row_order_validation.png")


### 3. Sample Size Sensitivity

**Purpose**: Assess how model performance varies with different training sample sizes to understand data requirements and diminishing returns.

**Approach**:
- Train models on varying percentages of training data (10%, 25%, 50%, 75%, 100%)
- Evaluate on a fixed test set
- Analyze performance curves to identify minimum viable sample size and saturation point

In [ ]:
print("Performing Sample Size Sensitivity Analysis...")

# Define sample size fractions to test
sample_fractions = [0.1, 0.25, 0.5, 1.0]  # Reduced fractions to save memory

# Store results
sample_size_results = {
    'sample_fraction': [],
    'n_samples': [],
    'lr_train_acc': [],
    'lr_test_acc': [],
    'lr_train_f1': [],
    'lr_test_f1': [],
    'lr_test_roc_auc': [],
    'nb_train_acc': [],
    'nb_test_acc': [],
    'nb_train_f1': [],
    'nb_test_f1': [],
    'nb_test_roc_auc': []
}

# Use fixed test set for consistency
X_test_sample = X_test_tfidf
y_test_sample = y_test

print(f"\nTraining models on {len(sample_fractions)} different sample sizes...")
print(f"Using fixed test set of {len(y_test_sample):,} samples\n")

for frac in sample_fractions:
    n_samples_use = int(frac * len(y_train))
    
    # Sample data (stratified to maintain class balance)
    if frac < 1.0:
        from sklearn.model_selection import train_test_split
        X_sample, _, y_sample, _ = train_test_split(
            X_train_tfidf, y_train,
            train_size=frac,
            stratify=y_train,
            random_state=42
        )
    else:
        X_sample = X_train_tfidf
        y_sample = y_train
    
    print(f"Training with {frac*100:>5.1f}% of data ({n_samples_use:>7,} samples)...", end=' ')
    
    # Train Logistic Regression
    lr_sample = LogisticRegression(random_state=42, max_iter=1000, 
                                     class_weight='balanced', solver='liblinear')
    lr_sample.fit(X_sample, y_sample)
    
    # Train Naive Bayes
    nb_sample = MultinomialNB()
    nb_sample.fit(X_sample.toarray(), y_sample)
    
    # Evaluate on training set
    lr_train_pred = lr_sample.predict(X_sample)
    lr_train_acc = accuracy_score(y_sample, lr_train_pred)
    lr_train_f1 = f1_score(y_sample, lr_train_pred, average='weighted')
    
    nb_train_pred = nb_sample.predict(X_sample.toarray())
    nb_train_acc = accuracy_score(y_sample, nb_train_pred)
    nb_train_f1 = f1_score(y_sample, nb_train_pred, average='weighted')
    
    # Evaluate on test set
    lr_test_pred = lr_sample.predict(X_test_sample)
    lr_test_acc = accuracy_score(y_test_sample, lr_test_pred)
    lr_test_f1 = f1_score(y_test_sample, lr_test_pred, average='weighted')
    lr_test_proba = lr_sample.predict_proba(X_test_sample)[:, 1]
    lr_test_roc = roc_auc_score(y_test_sample, lr_test_proba)
    
    nb_test_pred = nb_sample.predict(X_test_sample.toarray())
    nb_test_acc = accuracy_score(y_test_sample, nb_test_pred)
    nb_test_f1 = f1_score(y_test_sample, nb_test_pred, average='weighted')
    nb_test_proba = nb_sample.predict_proba(X_test_sample.toarray())[:, 1]
    nb_test_roc = roc_auc_score(y_test_sample, nb_test_proba)
    
    # Store results
    sample_size_results['sample_fraction'].append(frac)
    sample_size_results['n_samples'].append(n_samples_use)
    sample_size_results['lr_train_acc'].append(lr_train_acc)
    sample_size_results['lr_test_acc'].append(lr_test_acc)
    sample_size_results['lr_train_f1'].append(lr_train_f1)
    sample_size_results['lr_test_f1'].append(lr_test_f1)
    sample_size_results['lr_test_roc_auc'].append(lr_test_roc)
    sample_size_results['nb_train_acc'].append(nb_train_acc)
    sample_size_results['nb_test_acc'].append(nb_test_acc)
    sample_size_results['nb_train_f1'].append(nb_train_f1)
    sample_size_results['nb_test_f1'].append(nb_test_f1)
    sample_size_results['nb_test_roc_auc'].append(nb_test_roc)
    
    print(f"Done (LR: {lr_test_acc:.4f}, NB: {nb_test_acc:.4f})")

print("Sample size sensitivity analysis complete!")


In [ ]:
# Analyze sample size sensitivity results

sample_size_df = pd.DataFrame(sample_size_results)

print("\nSample Size Sensitivity Results")
print(f"{'Sample %':>10} {'N Samples':>12} {'LR Test Acc':>14} {'LR Test F1':>12} "
      f"{'NB Test Acc':>14} {'NB Test F1':>12} {'LR-NB Gap':>12}")

for _, row in sample_size_df.iterrows():
    lr_nb_gap = row['lr_test_acc'] - row['nb_test_acc']
    print(f"{row['sample_fraction']*100:>10.1f} {row['n_samples']:>12,} "
          f"{row['lr_test_acc']:>14.4f} {row['lr_test_f1']:>12.4f} "
          f"{row['nb_test_acc']:>14.4f} {row['nb_test_f1']:>12.4f} "
          f"{lr_nb_gap:>12.4f}")


# Calculate performance at different thresholds
print("\nPerformance Milestones:")

# Find sample size needed to reach 90% of full performance
lr_full_acc = sample_size_df['lr_test_acc'].iloc[-1]
nb_full_acc = sample_size_df['nb_test_acc'].iloc[-1]

lr_90pct = 0.90 * lr_full_acc
nb_90pct = 0.90 * nb_full_acc

lr_90_idx = sample_size_df[sample_size_df['lr_test_acc'] >= lr_90pct].index[0]
nb_90_idx = sample_size_df[sample_size_df['nb_test_acc'] >= nb_90pct].index[0]

print(f"\nTo reach 90% of full performance:")
print(f"  LR: {sample_size_df.iloc[lr_90_idx]['sample_fraction']*100:.1f}% "
      f"({sample_size_df.iloc[lr_90_idx]['n_samples']:,} samples)")
print(f"  NB: {sample_size_df.iloc[nb_90_idx]['sample_fraction']*100:.1f}% "
      f"({sample_size_df.iloc[nb_90_idx]['n_samples']:,} samples)")

# Calculate performance gains
print(f"\nPerformance gains from 25% to 100% of data:")
idx_25 = sample_size_df[sample_size_df['sample_fraction'] == 0.25].index[0]
idx_100 = sample_size_df[sample_size_df['sample_fraction'] == 1.0].index[0]

lr_gain = ((sample_size_df.iloc[idx_100]['lr_test_acc'] - 
            sample_size_df.iloc[idx_25]['lr_test_acc']) / 
           sample_size_df.iloc[idx_25]['lr_test_acc']) * 100

nb_gain = ((sample_size_df.iloc[idx_100]['nb_test_acc'] - 
            sample_size_df.iloc[idx_25]['nb_test_acc']) / 
           sample_size_df.iloc[idx_25]['nb_test_acc']) * 100

print(f"  LR: +{lr_gain:.2f}% improvement")
print(f"  NB: +{nb_gain:.2f}% improvement")

# Assess diminishing returns
print(f"\nPerformance gains from 50% to 100% of data:")  # Changed from 75% to 50%
idx_50 = sample_size_df[sample_size_df['sample_fraction'] == 0.5].index[0]  # Changed from 0.75 to 0.5

lr_gain_late = ((sample_size_df.iloc[idx_100]['lr_test_acc'] - 
                 sample_size_df.iloc[idx_50]['lr_test_acc']) /  # Changed from idx_75 to idx_50
                sample_size_df.iloc[idx_50]['lr_test_acc']) * 100

nb_gain_late = ((sample_size_df.iloc[idx_100]['nb_test_acc'] - 
                 sample_size_df.iloc[idx_50]['nb_test_acc']) /  # Changed from idx_75 to idx_50
                sample_size_df.iloc[idx_50]['nb_test_acc']) * 100

print(f"  LR: +{lr_gain_late:.2f}% improvement")
print(f"  NB: +{nb_gain_late:.2f}% improvement")

if lr_gain_late < 1.0 and nb_gain_late < 1.0:
    print("\nStrong diminishing returns observed - models saturate around 50% of data")  # Changed from 75%
elif lr_gain_late < 2.0 and nb_gain_late < 2.0:
    print("\nModerate diminishing returns - some benefit from additional data")
else:
    print("\nContinued strong gains - models could benefit from more data")


In [ ]:
# Visualize sample size sensitivity

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

# Plot 1: Accuracy vs Sample Size
axes[0, 0].plot(sample_size_df['n_samples'], sample_size_df['lr_train_acc'], 
                'o-', color='steelblue', alpha=0.5, label='LR Train', linewidth=2)
axes[0, 0].plot(sample_size_df['n_samples'], sample_size_df['lr_test_acc'], 
                'o-', color='steelblue', label='LR Test', linewidth=2)
axes[0, 0].plot(sample_size_df['n_samples'], sample_size_df['nb_train_acc'], 
                's-', color='coral', alpha=0.5, label='NB Train', linewidth=2)
axes[0, 0].plot(sample_size_df['n_samples'], sample_size_df['nb_test_acc'], 
                's-', color='coral', label='NB Test', linewidth=2)
axes[0, 0].set_xlabel('Training Sample Size', fontsize=11)
axes[0, 0].set_ylabel('Accuracy', fontsize=11)
axes[0, 0].set_title('Accuracy vs Training Sample Size', fontsize=12, fontweight='bold')
axes[0, 0].legend(fontsize=10)
axes[0, 0].grid(alpha=0.3)

# Plot 2: F1 Score vs Sample Size
axes[0, 1].plot(sample_size_df['n_samples'], sample_size_df['lr_train_f1'], 
                'o-', color='steelblue', alpha=0.5, label='LR Train', linewidth=2)
axes[0, 1].plot(sample_size_df['n_samples'], sample_size_df['lr_test_f1'], 
                'o-', color='steelblue', label='LR Test', linewidth=2)
axes[0, 1].plot(sample_size_df['n_samples'], sample_size_df['nb_train_f1'], 
                's-', color='coral', alpha=0.5, label='NB Train', linewidth=2)
axes[0, 1].plot(sample_size_df['n_samples'], sample_size_df['nb_test_f1'], 
                's-', color='coral', label='NB Test', linewidth=2)
axes[0, 1].set_xlabel('Training Sample Size', fontsize=11)
axes[0, 1].set_ylabel('F1 Score', fontsize=11)
axes[0, 1].set_title('F1 Score vs Training Sample Size', fontsize=12, fontweight='bold')
axes[0, 1].legend(fontsize=10)
axes[0, 1].grid(alpha=0.3)

# Plot 3: ROC-AUC vs Sample Size
axes[1, 0].plot(sample_size_df['n_samples'], sample_size_df['lr_test_roc_auc'], 
                'o-', color='steelblue', label='Logistic Regression', linewidth=2, markersize=8)
axes[1, 0].plot(sample_size_df['n_samples'], sample_size_df['nb_test_roc_auc'], 
                's-', color='coral', label='Naive Bayes', linewidth=2, markersize=8)
axes[1, 0].set_xlabel('Training Sample Size', fontsize=11)
axes[1, 0].set_ylabel('ROC-AUC (Test)', fontsize=11)
axes[1, 0].set_title('ROC-AUC vs Training Sample Size', fontsize=12, fontweight='bold')
axes[1, 0].legend(fontsize=10)
axes[1, 0].grid(alpha=0.3)

# Plot 4: Overfitting Gap (Train-Test) vs Sample Size
lr_gap = np.array(sample_size_df['lr_train_acc']) - np.array(sample_size_df['lr_test_acc'])
nb_gap = np.array(sample_size_df['nb_train_acc']) - np.array(sample_size_df['nb_test_acc'])

axes[1, 1].plot(sample_size_df['n_samples'], lr_gap, 
                'o-', color='steelblue', label='Logistic Regression', linewidth=2, markersize=8)
axes[1, 1].plot(sample_size_df['n_samples'], nb_gap, 
                's-', color='coral', label='Naive Bayes', linewidth=2, markersize=8)
axes[1, 1].axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)
axes[1, 1].set_xlabel('Training Sample Size', fontsize=11)
axes[1, 1].set_ylabel('Overfitting Gap (Train Acc - Test Acc)', fontsize=11)
axes[1, 1].set_title('Overfitting vs Training Sample Size', fontsize=12, fontweight='bold')
axes[1, 1].legend(fontsize=10)
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../Images/robustness_sample_size_sensitivity.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()  # Free memory  # Free memory

print("\nVisualization saved: ../Images/robustness_sample_size_sensitivity.png")

# Overfitting Gap = Training Accuracy − Test Accuracy

### Robustness Checks Summary

| Check Type | Purpose | Key Finding | Implication |
|------------|---------|-------------|-------------|
| **Cross-Validation** | Test consistency across data splits | Performance variance across 5 folds | Indicates reliability of results |
| **Row-Order Validation** | Test generalization to "future" data | Early→Recent performance comparison | Assesses row-order shift sensitivity |
| **Sample Size Sensitivity** | Test data requirements | Performance vs training size curve | Identifies minimum viable dataset |

**Overall Robustness Assessment:**
- **Low CV% (<5%)**: Models are highly robust to data partitioning
- **Stable row-order performance**: Models generalize well across time periods
- **Clear saturation point**: Identifies efficient data collection targets
- **Consistent LR vs NB ranking**: Results are not dependent on specific data splits

## Classification Diagnostics

These diagnostics assess whether our models are well-calibrated, how they trade off precision vs recall, and how performance scales with data size.

### 1. Calibration Curves

**Purpose**: Assess whether predicted probabilities match actual frequencies.

A well-calibrated model should have predictions where P(positive) = 0.7 means ~70% of those reviews are actually positive. Poor calibration means probability estimates are unreliable for decision-making.

**Why appropriate for our task**: If we use predictions for downstream decisions (e.g., flagging reviews for follow-up), we need to trust the probability estimates, not just the binary classifications.


### 2. ROC Curves and AUC

**Purpose**: Visualize the trade-off between True Positive Rate (sensitivity) and False Positive Rate across all classification thresholds.

**AUC (Area Under Curve)**: Probability that a randomly chosen positive example is ranked higher than a randomly chosen negative example. AUC = 1.0 is perfect; AUC = 0.5 is random guessing.

**Why appropriate**: ROC-AUC is threshold-independent, showing overall model discrimination ability regardless of the specific cutoff used.


### 3. Precision-Recall Curves

**Purpose**: For imbalanced datasets (our 2.6:1 ratio), PR curves are more informative than ROC curves because they focus on the positive class performance.

**Average Precision (AP)**: Summarizes the PR curve as the weighted mean of precisions at each threshold, with recall increase as weights. Higher is better.

**Why appropriate**: With class imbalance, high ROC-AUC can be misleading. PR curves reveal how well the model identifies positive reviews while maintaining precision.


In [ ]:
# Additional imports for diagnostics
from sklearn.calibration import calibration_curve
from sklearn.metrics import roc_curve, roc_auc_score, precision_recall_curve, average_precision_score


In [ ]:
# Get predicted probabilities for positive class
y_prob_lr = lr.predict_proba(X_test_tfidf)[:, 1] # Here 1 represents the positive class which means it'll calculate the proability of getting true prediction
y_prob_nb = nb.predict_proba(X_test_tfidf)[:, 1]

In [ ]:
# Compute calibration curves
# These lines compute calibration curves by binning predicted probabilities and comparing them with the observed fraction of 
# positive outcomes, enabling evaluation of how well the model’s probability estimates reflect true likelihoods.
# For a perfectly calibrated model: prob_true = prob_pred prob_true=prob_pred

prob_true_lr, prob_pred_lr = calibration_curve(y_test, y_prob_lr, n_bins=10, strategy='uniform')
prob_true_nb, prob_pred_nb = calibration_curve(y_test, y_prob_nb, n_bins=10, strategy='uniform')

In [ ]:
# Plot calibration curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Calibration plot
axes[0].plot([0, 1], [0, 1], 'k--', label='Perfectly Calibrated', linewidth=2)
axes[0].plot(prob_pred_lr, prob_true_lr, 's-', label='Logistic Regression', 
             color='steelblue', markersize=8, linewidth=2)
axes[0].plot(prob_pred_nb, prob_true_nb, 'o-', label='Naive Bayes', 
             color='coral', markersize=8, linewidth=2)
axes[0].set_xlabel('Mean Predicted Probability', fontweight='bold', fontsize=11)
axes[0].set_ylabel('Fraction of Positives', fontweight='bold', fontsize=11)
axes[0].set_title('Calibration Curves', fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Calibration error (Brier score proxy)
calib_error_lr = np.mean((prob_pred_lr - prob_true_lr) ** 2)
calib_error_nb = np.mean((prob_pred_nb - prob_true_nb) ** 2)

axes[1].bar(['Logistic Regression', 'Naive Bayes'], [calib_error_lr, calib_error_nb], 
            color=['steelblue', 'coral'], alpha=0.7)
axes[1].set_ylabel('Mean Squared Calibration Error', fontweight='bold', fontsize=11)
axes[1].set_title('Calibration Error (Lower = Better)', fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Compute ROC curves
fpr_lr, tpr_lr, thresholds_lr = roc_curve(y_test, y_prob_lr)
fpr_nb, tpr_nb, thresholds_nb = roc_curve(y_test, y_prob_nb)

# Compute AUC scores
auc_lr = roc_auc_score(y_test, y_prob_lr)
auc_nb = roc_auc_score(y_test, y_prob_nb)


In [ ]:
# Plot ROC curves
fig, ax = plt.subplots(figsize=(8, 8))

ax.plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC = {auc_lr:.4f})', 
        color='steelblue', linewidth=2)
ax.plot(fpr_nb, tpr_nb, label=f'Naive Bayes (AUC = {auc_nb:.4f})', 
        color='coral', linewidth=2)
ax.plot([0, 1], [0, 1], 'k--', label='Random Classifier (AUC = 0.5)', linewidth=1)

ax.set_xlabel('False Positive Rate (1 - Specificity)', fontweight='bold', fontsize=12)
ax.set_ylabel('True Positive Rate (Sensitivity/Recall)', fontweight='bold', fontsize=12)
ax.set_title('ROC Curves', fontweight='bold', fontsize=14)
ax.legend(loc='lower right')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Compute Precision-Recall curves
precision_lr, recall_lr, thresholds_pr_lr = precision_recall_curve(y_test, y_prob_lr)
precision_nb, recall_nb, thresholds_pr_nb = precision_recall_curve(y_test, y_prob_nb)

# Compute Average Precision
ap_lr = average_precision_score(y_test, y_prob_lr)
ap_nb = average_precision_score(y_test, y_prob_nb)

# Baseline (random classifier) for imbalanced data
baseline = y_test.mean()  # Proportion of positive class


In [ ]:
# Plot PR curves
fig, ax = plt.subplots(figsize=(8, 8))

ax.plot(recall_lr, precision_lr, label=f'Logistic Regression (AP = {ap_lr:.4f})', 
        color='steelblue', linewidth=2)
ax.plot(recall_nb, precision_nb, label=f'Naive Bayes (AP = {ap_nb:.4f})', 
        color='coral', linewidth=2)
ax.axhline(y=baseline, color='gray', linestyle='--', 
           label=f'Baseline (Positive Rate = {baseline:.3f})')

ax.set_xlabel('Recall', fontweight='bold', fontsize=12)
ax.set_ylabel('Precision', fontweight='bold', fontsize=12)
ax.set_title('Precision-Recall Curves', fontweight='bold', fontsize=14)
ax.legend(loc='upper right')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()


### Classification Diagnostics Summary
| Diagnostic | What It Reveals | Why It Matters |
|------------|-----------------|----------------|
| **Calibration** | Reliability of predicted probabilities | Needed for probability-based decisions |
| **ROC-AUC** | Overall discrimination ability | Threshold-independent comparison |
| **Precision-Recall** | Positive-class performance under imbalance | More informative for our class ratio |
**Key Findings:**
1. Both models are reasonably calibrated, so probability estimates are usable.
2. High ROC-AUC indicates strong separation between positive and negative reviews.
3. PR curves confirm performance on the minority class is solid, not just overall accuracy.


## Error Analysis

A detailed analysis of misclassifications to understand model limitations and error patterns.

In [ ]:
# Error Analysis: Categorize and analyze misclassifications
import re
import pandas as pd
import numpy as np

# Get misclassified indices for both models
errors_lr = y_test != y_test_pred_lr
errors_nb = y_test != y_test_pred_nb

# Get confidence scores (max probability across classes)
# Since y_prob_lr and y_prob_nb are 1D (probability of positive class only),
# we need to calculate confidence as max(prob, 1-prob)
confidence_lr = np.maximum(y_prob_lr, 1 - y_prob_lr)
confidence_nb = np.maximum(y_prob_nb, 1 - y_prob_nb)

# Get the actual text data for test set
test_texts = X_test.values

print("ERROR ANALYSIS")

# Function to categorize error types
def categorize_error(text):
    text_lower = text.lower()
    categories = []
    
    # Check for negation patterns
    negation_patterns = [r"\bnot\b", r"\bno\b", r"\bn't\b", r"\bnever\b", 
                        r"\bneither\b", r"\bnor\b", r"\bhardly\b", r"\bbarely\b"]
    if any(re.search(pattern, text_lower) for pattern in negation_patterns):
        categories.append('negation')
    
    # Check for sarcasm/irony indicators
    sarcasm_patterns = [r"\breally\b.*\bgood\b", r"\bsure\b", r"\byeah right\b",
                       r"\boh great\b", r"\bwonderful\b.*\bnot\b", r"\btotally\b"]
    if any(re.search(pattern, text_lower) for pattern in sarcasm_patterns):
        categories.append('sarcasm')
    
    # Check for mixed sentiment (both positive and negative words)
    positive_words = ['good', 'great', 'excellent', 'love', 'like', 'best', 'helpful']
    negative_words = ['bad', 'terrible', 'awful', 'hate', 'worst', 'poor', 'confusing']
    
    has_positive = any(word in text_lower for word in positive_words)
    has_negative = any(word in text_lower for word in negative_words)
    
    if has_positive and has_negative:
        categories.append('mixed_sentiment')
    
    if not categories:
        categories.append('other')
    
    return categories

# Analyze errors for Logistic Regression
print("LOGISTIC REGRESSION - ERROR CATEGORIZATION")

lr_error_texts = test_texts[errors_lr]
lr_error_true = y_test[errors_lr]
lr_error_pred = y_test_pred_lr[errors_lr]
lr_error_conf = confidence_lr[errors_lr]

# Categorize LR errors
lr_error_categories = {'negation': 0, 'sarcasm': 0, 'mixed_sentiment': 0, 'other': 0}
for text in lr_error_texts:
    cats = categorize_error(text)
    for cat in cats:
        lr_error_categories[cat] += 1

print(f"\nTotal Errors: {len(lr_error_texts):,}")
print(f"\nError Type Distribution:")
for cat, count in sorted(lr_error_categories.items(), key=lambda x: x[1], reverse=True):
    pct = (count / len(lr_error_texts)) * 100
    print(f"  {cat.replace('_', ' ').title():20s}: {count:4d} ({pct:5.1f}%)")

# Analyze errors for Naive Bayes
print("NAIVE BAYES - ERROR CATEGORIZATION")

nb_error_texts = test_texts[errors_nb]
nb_error_true = y_test[errors_nb]
nb_error_pred = y_test_pred_nb[errors_nb]
nb_error_conf = confidence_nb[errors_nb]

# Categorize NB errors
nb_error_categories = {'negation': 0, 'sarcasm': 0, 'mixed_sentiment': 0, 'other': 0}
for text in nb_error_texts:
    cats = categorize_error(text)
    for cat in cats:
        nb_error_categories[cat] += 1

print(f"\nTotal Errors: {len(nb_error_texts):,}")
print(f"\nError Type Distribution:")
for cat, count in sorted(nb_error_categories.items(), key=lambda x: x[1], reverse=True):
    pct = (count / len(nb_error_texts)) * 100
    print(f"  {cat.replace('_', ' ').title():20s}: {count:4d} ({pct:5.1f}%)")

# Visualize error type distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# LR error distribution
lr_cats = list(lr_error_categories.keys())
lr_vals = [lr_error_categories[cat] for cat in lr_cats]
axes[0].bar(range(len(lr_cats)), lr_vals, color='steelblue', alpha=0.7)
axes[0].set_xticks(range(len(lr_cats)))
axes[0].set_xticklabels([cat.replace('_', '\n').title() for cat in lr_cats], rotation=0)
axes[0].set_ylabel('Number of Errors', fontweight='bold')
axes[0].set_title('Logistic Regression', fontweight='bold', fontsize=12)
axes[0].grid(axis='y', alpha=0.3)

# NB error distribution
nb_cats = list(nb_error_categories.keys())
nb_vals = [nb_error_categories[cat] for cat in nb_cats]
axes[1].bar(range(len(nb_cats)), nb_vals, color='darkorange', alpha=0.7)
axes[1].set_xticks(range(len(nb_cats)))
axes[1].set_xticklabels([cat.replace('_', '\n').title() for cat in nb_cats], rotation=0)
axes[1].set_ylabel('Number of Errors', fontweight='bold')
axes[1].set_title('Naive Bayes', fontweight='bold', fontsize=12)
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('Error Type Distribution by Model', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../Images/error_type_distribution.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()  # Free memory

# Confidence Analysis: Compare confidence on correct vs incorrect predictions


In [ ]:
print("CONFIDENCE ANALYSIS")

# Logistic Regression confidence analysis
lr_correct = ~errors_lr
lr_conf_correct = confidence_lr[lr_correct]
lr_conf_error = confidence_lr[errors_lr]

print("\nLogistic Regression:")
print(f"  Confidence on Correct Predictions:")
print(f"    Mean: {lr_conf_correct.mean():.3f}")
print(f"    Std:  {lr_conf_correct.std():.3f}")
print(f"    Min:  {lr_conf_correct.min():.3f}")
print(f"    Max:  {lr_conf_correct.max():.3f}")

print(f"\n  Confidence on Incorrect Predictions:")
print(f"    Mean: {lr_conf_error.mean():.3f}")
print(f"    Std:  {lr_conf_error.std():.3f}")
print(f"    Min:  {lr_conf_error.min():.3f}")
print(f"    Max:  {lr_conf_error.max():.3f}")

# Naive Bayes confidence analysis
nb_correct = ~errors_nb
nb_conf_correct = confidence_nb[nb_correct]
nb_conf_error = confidence_nb[errors_nb]

print("\nNaive Bayes:")
print(f"  Confidence on Correct Predictions:")
print(f"    Mean: {nb_conf_correct.mean():.3f}")
print(f"    Std:  {nb_conf_correct.std():.3f}")
print(f"    Min:  {nb_conf_correct.min():.3f}")
print(f"    Max:  {nb_conf_correct.max():.3f}")

print(f"\n  Confidence on Incorrect Predictions:")
print(f"    Mean: {nb_conf_error.mean():.3f}")
print(f"    Std:  {nb_conf_error.std():.3f}")
print(f"    Min:  {nb_conf_error.min():.3f}")
print(f"    Max:  {nb_conf_error.max():.3f}")

# Visualize confidence distributions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# LR confidence histogram
axes[0, 0].hist(lr_conf_correct, bins=30, alpha=0.6, label='Correct', color='green', edgecolor='black')
axes[0, 0].hist(lr_conf_error, bins=30, alpha=0.6, label='Incorrect', color='red', edgecolor='black')
axes[0, 0].set_xlabel('Confidence Score', fontweight='bold')
axes[0, 0].set_ylabel('Frequency', fontweight='bold')
axes[0, 0].set_title('Logistic Regression - Confidence Distribution', fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# NB confidence histogram
axes[0, 1].hist(nb_conf_correct, bins=30, alpha=0.6, label='Correct', color='green', edgecolor='black')
axes[0, 1].hist(nb_conf_error, bins=30, alpha=0.6, label='Incorrect', color='red', edgecolor='black')
axes[0, 1].set_xlabel('Confidence Score', fontweight='bold')
axes[0, 1].set_ylabel('Frequency', fontweight='bold')
axes[0, 1].set_title('Naive Bayes - Confidence Distribution', fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# LR box plot
axes[1, 0].boxplot([lr_conf_correct, lr_conf_error], 
                   labels=['Correct', 'Incorrect'],
                   patch_artist=True,
                   boxprops=dict(facecolor='lightblue', alpha=0.7),
                   medianprops=dict(color='red', linewidth=2))
axes[1, 0].set_ylabel('Confidence Score', fontweight='bold')
axes[1, 0].set_title('Logistic Regression - Confidence Comparison', fontweight='bold')
axes[1, 0].grid(axis='y', alpha=0.3)

# NB box plot
axes[1, 1].boxplot([nb_conf_correct, nb_conf_error], 
                   labels=['Correct', 'Incorrect'],
                   patch_artist=True,
                   boxprops=dict(facecolor='lightcoral', alpha=0.7),
                   medianprops=dict(color='darkred', linewidth=2))
axes[1, 1].set_ylabel('Confidence Score', fontweight='bold')
axes[1, 1].set_title('Naive Bayes - Confidence Comparison', fontweight='bold')
axes[1, 1].grid(axis='y', alpha=0.3)

plt.suptitle('Confidence Analysis: Correct vs Incorrect Predictions', 
             fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('../Images/confidence_analysis.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()  # Free memory  # Free memory

# Statistical test: t-test for confidence differences
from scipy import stats

lr_t_stat, lr_p_value = stats.ttest_ind(lr_conf_correct, lr_conf_error)
nb_t_stat, nb_p_value = stats.ttest_ind(nb_conf_correct, nb_conf_error)

print("Statistical Significance of Confidence Differences")
print(f"\nLogistic Regression:")
print(f"  t-statistic: {lr_t_stat:.4f}")
print(f"  p-value:     {lr_p_value:.4e}")
print(f"  Significant: {'Yes' if lr_p_value < 0.05 else 'No'} (α=0.05)")

print(f"\nNaive Bayes:")
print(f"  t-statistic: {nb_t_stat:.4f}")
print(f"  p-value:     {nb_p_value:.4e}")
print(f"  Significant: {'Yes' if nb_p_value < 0.05 else 'No'} (α=0.05)")

# Show example misclassifications


In [ ]:
print("EXAMPLE MISCLASSIFICATIONS")

# Function to display error examples
def show_error_examples(texts, true_labels, pred_labels, confidences, model_name, n_examples=5):
    print(f"\n{model_name}:")
    
    # Convert to numpy arrays for positional indexing
    texts = np.array(texts)
    true_labels = np.array(true_labels)
    pred_labels = np.array(pred_labels)
    confidences = np.array(confidences)
    
    # Sort by confidence (show most confident mistakes)
    sorted_indices = np.argsort(confidences)[::-1]
    
    for i, idx in enumerate(sorted_indices[:n_examples]):
        text = texts[idx]
        true_label = 'Positive' if true_labels[idx] == 1 else 'Negative'
        pred_label = 'Positive' if pred_labels[idx] == 1 else 'Negative'
        conf = confidences[idx]
        
        # Categorize error
        error_cats = categorize_error(text)
        
        print(f"\nExample {i+1}:")
        print(f"  Text: {text[:200]}{'...' if len(text) > 200 else ''}")
        print(f"  True Label:      {true_label}")
        print(f"  Predicted Label: {pred_label}")
        print(f"  Confidence:      {conf:.3f}")
        print(f"  Error Type:      {', '.join(error_cats)}")

# Show LR examples
show_error_examples(lr_error_texts, lr_error_true, lr_error_pred, 
                   lr_error_conf, "Logistic Regression", n_examples=5)

# Show NB examples
show_error_examples(nb_error_texts, nb_error_true, nb_error_pred, 
                   nb_error_conf, "Naive Bayes", n_examples=5)


### Supplemental Negation Experiment (Stats 201-Informed)

This controlled experiment adds a negation-scope preprocessing rule (`not good -> NEG_good`) and compares it against the baseline Logistic Regression pipeline.
It reports baseline vs negation-aware F1 and negation-tagged error reduction.

In [ ]:
# Supplemental experiment: negation-scope preprocessing for Logistic Regression
import re
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
import pathlib

negation_summary_path = WEEK6_DIR / 'negation_experiment_summary.csv' if 'WEEK6_DIR' in globals() else pathlib.Path('./outputs/week6/negation_experiment_summary.csv')

if negation_summary_path.exists():
    negation_summary_df = pd.read_csv(negation_summary_path)
    print('Loaded existing negation experiment summary:')
    display(negation_summary_df)
else:
    neg_re = re.compile(r"\b(?:not|no|never|neither|nor|hardly|barely|n't)\b", re.IGNORECASE)
    tok_re = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?")

    def mark_negation(text, window=2):
        tokens = tok_re.findall(str(text))
        out = []
        for i, tok in enumerate(tokens):
            out.append(tok)
            if neg_re.fullmatch(tok.lower()):
                for j in range(1, window + 1):
                    if i + j < len(tokens):
                        out.append('NEG_' + tokens[i + j].lower())
        return ' '.join(out)

    combined_stopwords = list(ENGLISH_STOP_WORDS.union(domain_stopwords))

    # Baseline
    tfidf_base = TfidfVectorizer(
        max_features=10000, ngram_range=(1, 2), min_df=5, max_df=0.8,
        stop_words=combined_stopwords, token_pattern=r'\b[a-zA-Z]{3,}\b'
    )
    X_train_base = tfidf_base.fit_transform(X_train)
    X_test_base = tfidf_base.transform(X_test)
    lr_base = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced', C=1.0, solver='liblinear')
    lr_base.fit(X_train_base, y_train)
    pred_base = lr_base.predict(X_test_base)

    # Negation-aware
    X_train_neg = X_train.map(mark_negation)
    X_test_neg = X_test.map(mark_negation)
    tfidf_neg = TfidfVectorizer(
        max_features=10000, ngram_range=(1, 2), min_df=5, max_df=0.8,
        stop_words=combined_stopwords, token_pattern=r'\b[a-zA-Z_]{3,}\b'
    )
    X_train_neg_tfidf = tfidf_neg.fit_transform(X_train_neg)
    X_test_neg_tfidf = tfidf_neg.transform(X_test_neg)
    lr_neg = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced', C=1.0, solver='liblinear')
    lr_neg.fit(X_train_neg_tfidf, y_train)
    pred_neg = lr_neg.predict(X_test_neg_tfidf)

    neg_mask = X_test.str.contains(r"\b(not|no|never|neither|nor|hardly|barely|n't)\b", case=False, regex=True, na=False)
    base_neg_err = int((pred_base[neg_mask.values] != y_test[neg_mask].values).sum())
    neg_model_err = int((pred_neg[neg_mask.values] != y_test[neg_mask].values).sum())

    negation_summary_df = pd.DataFrame([
        {
            'test_n': int(len(y_test)),
            'negation_subset_n': int(neg_mask.sum()),
            'baseline_accuracy': round(float(accuracy_score(y_test, pred_base)), 4),
            'baseline_f1': round(float(f1_score(y_test, pred_base)), 4),
            'negation_aware_accuracy': round(float(accuracy_score(y_test, pred_neg)), 4),
            'negation_aware_f1': round(float(f1_score(y_test, pred_neg)), 4),
            'delta_f1': round(float(f1_score(y_test, pred_neg) - f1_score(y_test, pred_base)), 4),
            'baseline_negation_errors': base_neg_err,
            'negation_aware_negation_errors': neg_model_err,
            'negation_error_reduction_count': base_neg_err - neg_model_err,
            'negation_error_reduction_pct': round(((base_neg_err - neg_model_err) / base_neg_err * 100) if base_neg_err else 0.0, 2),
        }
    ])

    out_dir = pathlib.Path('./outputs/week6')
    out_dir.mkdir(parents=True, exist_ok=True)
    negation_summary_df.to_csv(out_dir / 'negation_experiment_summary.csv', index=False)
    negation_summary_df.to_json(out_dir / 'negation_experiment_summary.json', orient='records', indent=2)
    print('Saved negation experiment summary to outputs/week6/.')
    display(negation_summary_df)

### Final Reporting Setup

This cell initializes shared imports, paths, and helper utilities used by the Week 6 synthesis workflow. It creates `outputs/final/` and `outputs/final/figures/` for generated artifacts.


In [ ]:
# Final reporting setup: shared paths and utilities
from pathlib import Path
from datetime import datetime
import json
import sys
import platform
import subprocess
import importlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

WEEK6_DIR = Path("./outputs/week6")
FIG_DIR = WEEK6_DIR / "figures"
WEEK6_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"Week 6 artifacts will be written to: {WEEK6_DIR.resolve()}")


def pick_first_defined(names):
    for name in names:
        if name in globals() and globals()[name] is not None:
            return name, globals()[name]
    return None, None


### Final Analytic Configuration and Stopping Justification

This cell captures the selected model, key parameters, and available performance indicators, then writes a machine-readable summary to `outputs/final/final_analytic_configuration.json`.


In [ ]:
# 1) Final analytic configuration + stopping justification
candidate_model_names = [
    "best_model", "final_model", "model", "clf", "classifier", "best_estimator", "best_estimator_"
]
model_name, model_obj = pick_first_defined(candidate_model_names)

params = {}
if model_obj is not None and hasattr(model_obj, "get_params"):
    try:
        params = model_obj.get_params()
    except Exception:
        params = {}

metric_candidates = [
    "best_score", "best_score_", "cv_best_score", "test_f1", "test_accuracy"
]
metrics_snapshot = {}
for key in metric_candidates:
    if key in globals() and globals()[key] is not None:
        value = globals()[key]
        if isinstance(value, (np.floating, np.integer)):
            value = float(value)
        metrics_snapshot[key] = value

stopping_rule = {
    "criterion": "Stop model iteration when test performance improvement is <= 0.005 for two consecutive model updates, or when additional complexity hurts generalization.",
    "evidence": metrics_snapshot if metrics_snapshot else "No tracked test metric variables were detected; use model comparison table already produced above.",
}

total_n = int(len(df_final)) if "df_final" in globals() else None
positive_n = int((df_final["sentiment"] == 1).sum()) if "df_final" in globals() else None
negative_n = int((df_final["sentiment"] == 0).sum()) if "df_final" in globals() else None

multiclass_sensitivity = None
try:
    sensitivity_path = WEEK6_DIR / "sensitivity_multiclass_summary.json"
    if sensitivity_path.exists():
        multiclass_sensitivity = json.loads(sensitivity_path.read_text())
except Exception:
    multiclass_sensitivity = None

final_config = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "selected_model_variable": model_name,
    "selected_model_class": type(model_obj).__name__ if model_obj is not None else None,
    "selected_model_params": params,
    "dependent_variable": {
        "name": "binary_sentiment",
        "coding": "Y=1 for ratings {4,5}; Y=0 for ratings {1,2}; rating 3 excluded",
        "class_counts": {"positive": positive_n, "negative": negative_n, "total": total_n},
        "class_percentages": {
            "positive_pct": round((positive_n / total_n) * 100, 2) if total_n else None,
            "negative_pct": round((negative_n / total_n) * 100, 2) if total_n else None
        },
        "rationale": "Binary coding improves interpretability/actionability and reduces midpoint ambiguity; granularity loss is tracked through a non-binary sensitivity check."
    },
    "non_binary_sensitivity_check": multiclass_sensitivity,
    "stopping_justification": stopping_rule,
}

config_path = WEEK6_DIR / "final_analytic_configuration.json"
with open(config_path, "w") as f:
    json.dump(final_config, f, indent=2, default=str)

print(f"Saved: {config_path}")
print(pd.Series({
    "selected_model_variable": final_config["selected_model_variable"],
    "selected_model_class": final_config["selected_model_class"],
    "num_params": len(final_config["selected_model_params"]),
}))


### Substantive Interpretation of Results

This cell builds a draft interpretation table connecting the research question to core metrics and domain-level takeaways. It exports a report-ready draft to `outputs/final/substantive_interpretation_draft.csv`.


### Limitations and Scope

This cell creates a structured draft of methodological limitations, why they matter, scope boundaries, and concrete mitigation next steps. Output is saved to `outputs/final/limitations_and_scope_draft.csv`.


In [ ]:
# 3) Limitations and scope table for the written report
neg_summary = None
neg_path = WEEK6_DIR / "negation_experiment_summary.csv"
if neg_path.exists():
    try:
        neg_summary = pd.read_csv(neg_path).iloc[0].to_dict()
    except Exception:
        neg_summary = None

negation_note = "Add and evaluate a negation-scope preprocessing rule (not good -> NEG_good), then compare sequence-aware models."
if neg_summary:
    negation_note = (
        f"Negation-scope test improved F1 by {neg_summary.get('delta_f1', 0):.4f} "
        f"and reduced negation-tagged errors by {neg_summary.get('negation_error_reduction_pct', 0):.2f}%."
    )

limitations_df = pd.DataFrame([
    {
        "limitation": "Potential class imbalance and under-representation",
        "why_it_matters": "Can inflate aggregate metrics while underperforming on minority cases",
        "scope_boundary": "Findings are strongest for classes with adequate training samples",
        "mitigation_next_step": "Report per-class metrics and evaluate re-sampling/cost-sensitive learning",
    },
    {
        "limitation": "Causal inference limitation (observational design)",
        "why_it_matters": "Unobserved confounding and selection effects limit causal identification",
        "scope_boundary": "Interpret coefficients as predictive associations, not causal effects",
        "mitigation_next_step": "Use quasi-experimental/panel designs for causal claims",
    },
    {
        "limitation": "External validity limitation (single-platform context)",
        "why_it_matters": "Patterns from RateMyProfessors may not transfer to all institutions/platforms",
        "scope_boundary": "Generalization is strongest for similar review environments",
        "mitigation_next_step": "Validate on external datasets from other educational settings",
    },
    {
        "limitation": "Model tuning search budget is finite",
        "why_it_matters": "Global optimum may not be reached",
        "scope_boundary": "Best model is conditional on tested algorithms and hyperparameter ranges",
        "mitigation_next_step": "Expand search space only if expected gain justifies compute cost",
    },
    {
        "limitation": "Negation sensitivity in bag-of-words representation",
        "why_it_matters": "Negation can invert polarity and create high-confidence mistakes",
        "scope_boundary": "Standard TF-IDF features under-represent local word order effects",
        "mitigation_next_step": negation_note,
    },
])

limits_path = WEEK6_DIR / "limitations_and_scope_draft.csv"
limitations_df.to_csv(limits_path, index=False)
print(f"Saved: {limits_path}")
limitations_df


### Draft Figures and Reproducibility Artifacts

This cell generates presentation-ready draft figures (when prediction variables exist) and writes reproducibility files, including environment snapshot and artifact manifest.


In [ ]:
# 4) Draft presentation figures + 5) reproducibility artifacts
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Figure A: core metric bars (if available)
if "metric_summary" in globals() and metric_summary:
    fig, ax = plt.subplots(figsize=(7, 4))
    keys = list(metric_summary.keys())
    vals = [metric_summary[k] for k in keys]
    sns.barplot(x=keys, y=vals, ax=ax, palette="Blues_d")
    ax.set_ylim(0, 1)
    ax.set_title("Draft Figure: Core Evaluation Metrics")
    ax.set_ylabel("Score")
    ax.set_xlabel("")
    plt.xticks(rotation=20)
    plt.tight_layout()
    p = FIG_DIR / "draft_core_metrics.png"
    fig.savefig(p, dpi=200)
    plt.show()
    print(f"Saved: {p}")

# Figure B: confusion matrix (if labels are available)
if "y_true" in globals() and "y_pred" in globals() and y_true is not None and y_pred is not None:
    fig, ax = plt.subplots(figsize=(6, 6))
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title("Draft Figure: Confusion Matrix")
    plt.tight_layout()
    p = FIG_DIR / "draft_confusion_matrix.png"
    fig.savefig(p, dpi=200)
    plt.show()
    print(f"Saved: {p}")

# Reproducibility snapshot
requirements_path = WEEK6_DIR / "requirements_snapshot.txt"
try:
    reqs = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
    requirements_path.write_text(reqs)
    print(f"Saved: {requirements_path}")
except Exception as e:
    print(f"Could not create requirements snapshot: {e}")

manifest = {
    "generated_at": datetime.now().isoformat(timespec="seconds"),
    "python_version": sys.version,
    "platform": platform.platform(),
    "artifacts": sorted([str(p) for p in WEEK6_DIR.rglob("*") if p.is_file()]),
}

manifest_path = WEEK6_DIR / "reproducibility_manifest.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

print(f"Saved: {manifest_path}")
print("Week 6 synthesis block completed.")
